In [1]:
import itertools
import pathlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from scipy.stats import permutation_test, friedmanchisquare

from sklearn.metrics import average_precision_score, f1_score, classification_report, accuracy_score

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.transforms import v2, InterpolationMode
from utils_data import NormTransform, RandomWrapAround

import torchvision.models as models
import timm

# Set matplotlib font
import matplotlib
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['font.size'] = 16


## Load mapping dfs

### Lab dataset

In [ ]:
map_df_paths = list(pathlib.Path(r"C:\Users\Ana-Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20").glob('*.csv'))
map_df_paths = [p for p in map_df_paths if '_Classes3_' in p.__str__()]
[print(p) for p in map_df_paths]
lab_dfs = pd.concat([pd.read_csv(p) for p in map_df_paths], axis=0).copy()
lab_dfs['img_relative_path'] = [pathlib.Path(p) for p in lab_dfs['img_relative_path']]
lab_dfs.loc[:, 'area'] = [p.parent.name.split('_')[-1] for p in lab_dfs['img_relative_path']]
lab_dfs

### Clinical dataset

In [ ]:
# Clinical data
map_df_paths = list(pathlib.Path(r"Y:\OCT_clinical_data\512mscans_sample20").glob('*.csv'))
map_df_paths = [p for p in map_df_paths if 'old' not in p.__str__()]
[print(p) for p in map_df_paths]
cli_dfs = pd.concat([pd.read_csv(p) for p in map_df_paths], axis=0)
cli_dfs['img_relative_path'] = [pathlib.Path(p) for p in cli_dfs['img_relative_path']]
cli_dfs.loc[:, 'area'] = [p.parent.name.split('_')[-1] for p in cli_dfs['img_relative_path']]
# cli_dfs.loc[:, 'pat'] = [int(re.sub(r'[^\d]+', '', p.parts[1])) for p in cli_dfs['img_relative_path']]
cli_dfs.loc[:, 'trajectory'] = ['_'.join(p.stem.split('_')[:-3]) for p in cli_dfs.loc[:, 'img_relative_path']]
cli_dfs

## Obsolete

### Check label and area distribution in lab test set

In [ ]:
lab_test_map = pd.read_csv(r"C:\Users\anaja\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20\test_mapping_512scans.csv")
ascan_per_group = lab_test_map['idx_end'].iloc[0]
lab_test_map.loc[:, 'img_relative_path'] = [pathlib.Path(p) for p in lab_test_map.loc[:,'img_relative_path']]
lab_test_map = lab_test_map.rename(columns={'label': 'label_str'})
labels_dict = {i: lbl for i, lbl in enumerate(['chicken_heart', 'chicken_stomach', 'lamb_heart', 'lamb_testicle'])}
for i, lbl in labels_dict.items():
   lab_test_map.loc[lab_test_map['label_str'] == lbl, 'label'] = i
lab_test_map['area'] = [re.sub('|'.join(f'{f}_' for f in labels_dict.values()), '', p.parts[1]) for p in lab_test_map['img_relative_path']]
lab_test_map['trajectory'] = ['_'.join(p.stem.split('_')[:-2]) for p in lab_test_map['img_relative_path']]
lab_test_map['area_id'] = lab_test_map.groupby(['label', 'area']).ngroup()

# Add traj_type
for i, traj_type in enumerate(['jitter', 'line', 'sine']):
    lab_test_map.loc[lab_test_map['trajectory'].str.contains(traj_type), 'traj_type'] = i

# Assign subset
# lab_test_map.loc[:, 'subset'] = -1
# lab_test_map.loc[lab_test_map.index%5 <= 2, 'subset'] = 'test_train'
# lab_test_map.loc[lab_test_map.index%5 == 3, 'subset'] = 'test_valid'
# lab_test_map.loc[lab_test_map.index%5 == 4, 'subset'] = 'test_test'
    
# Assign subset V2
lab_test_map.loc[:, 'subset'] = lab_test_map.groupby(['area_id', 'trajectory']).cumcount()
lab_test_map.loc[lab_test_map['subset']%10 == 9, 'subset'] = 'test_supervised'
lab_test_map.loc[lab_test_map['subset'] != 'test_supervised', 'subset'] = 'test'

lab_test_map

In [ ]:
lab_test_map.groupby(['area_id', 'traj_type', 'subset']).agg(img_count=('img_relative_path', 'count'), idx_start_mean=('idx_start', 'mean'))

In [ ]:
subset = 'test_supervised'
lab_test_map[lab_test_map['subset'] == f'test_{subset}']

In [ ]:
lab_test_map.groupby(['subset']).agg(img_count=('img_relative_path', 'count'), idx_start_mean=('idx_start', 'mean'))

### Plot results V0

In [ ]:
results_simclr = {'ResNet18': {'Supervised': {'F1 score': 48, 'Accuracy': 46}, 'Finetune': {'F1 score': 50, 'Accuracy': 49}},
           'EfficientNetV2-S': {'Supervised': {'F1 score': 53, 'Accuracy': 52}, 'Finetune': {'F1 score': 55, 'Accuracy': 53}}, 
           'ConvNeXt-Tiny': {'Supervised': {'F1 score': 49, 'Accuracy': 47}, 'Finetune': {'F1 score': 53, 'Accuracy': 52}}, 
           'PVT-V2-B0': {'Supervised': {'F1 score': 56, 'Accuracy': 55}, 'Finetune': {'F1 score': 51, 'Accuracy': 52}}}


results_byol = {'ResNet18': {'Supervised': {'F1 score': 48, 'Accuracy': 46}, 'Finetune': {'F1 score': 51, 'Accuracy': 51}},
           'EfficientNetV2-S': {'Supervised': {'F1 score': 53, 'Accuracy': 52}, 'Finetune': {'F1 score': 45, 'Accuracy': 47}}, 
           'ConvNeXt-Tiny': {'Supervised': {'F1 score': 49, 'Accuracy': 47}, 'Finetune': {'F1 score': 55, 'Accuracy': 53}}, 
           'PVT-V2-B0': {'Supervised': {'F1 score': 56, 'Accuracy': 55}, 'Finetune': {'F1 score': 57, 'Accuracy': 57}}}

results_simclr_df = pd.concat({m: pd.DataFrame.from_dict(res, orient='columns') for m, res in results_simclr.items()}).reset_index().rename(columns={'level_0': 'Model', 'level_1': 'Metric'})
results_simclr_df = pd.pivot_table(results_simclr_df, values=['Supervised', 'Finetune'], index='Model', columns='Metric').swaplevel(axis=1)
results_simclr_df.columns.set_names('Train type', level=1, inplace=True)
# results_simclr_df
# results_simclr_df['Accuracy'].plot.bar(rot=0, title='', ylabel='Accuracy')

In [ ]:
f1_results = {'ResNet18': {'Supervised': 48, 'Finetune-SimCLR': 50, 'Finetune-BYOL': 51},
           'EfficientNetV2-S': {'Supervised': 53, 'Finetune-SimCLR': 55, 'Finetune-BYOL': 45}, 
           'ConvNeXt-Tiny': {'Supervised': 49, 'Finetune-SimCLR': 53, 'Finetune-BYOL': 55}, 
           'PVT-V2-B0': {'Supervised': 56, 'Finetune-SimCLR': 51, 'Finetune-BYOL': 57}}
acc_results = {'ResNet18': {'Supervised': 46, 'Finetune-SimCLR': 49, 'Finetune-BYOL': 51},
           'EfficientNetV2-S': {'Supervised': 52, 'Finetune-SimCLR': 53, 'Finetune-BYOL': 47}, 
           'ConvNeXt-Tiny': {'Supervised': 47, 'Finetune-SimCLR': 52, 'Finetune-BYOL': 53}, 
           'PVT-V2-B0': {'Supervised': 55, 'Finetune-SimCLR': 52, 'Finetune-BYOL': 57}}
f1_results = pd.DataFrame.from_dict(f1_results, orient='index')
acc_results = pd.DataFrame.from_dict(acc_results, orient='index')

In [ ]:
ax_f1 = f1_results.plot.bar(rot=0, ylabel='F1 score', title='Training method comparison - F1 score')
ax_f1.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))

In [ ]:
ax_acc = acc_results.plot.bar(rot=0, ylabel='Accuracy', title='Training method comparison - Accuracy')
ax_acc.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))

## Lab dataset stats

In [ ]:
print(f"Total image count: {lab_dfs.groupby(['label']).agg(img_count=('img_relative_path', 'count')).sum()}")
lab_dfs.groupby(['label']).agg(img_count=('img_relative_path', 'count'))

In [ ]:
pd.merge(lab_dfs.groupby(['label', 'area']).agg(img_count=('img_relative_path', 'count')).reset_index(), lab_dfs[['label', 'old_label', 'area']].drop_duplicates(subset=['label', 'area'])).sort_values(['label', 'area'])

In [ ]:
lab_dfs.groupby(['label', 'subset']).agg(img_count=('img_relative_path', 'count'))

## Get clinical data stats

### Double check labels

In [ ]:
lbl_paths = list(pathlib.Path(r'Y:\OCT_clinical_data\Labels').rglob('*.csv'))
lbls = {'_'.join(p.stem.split('_')[:-2]): pd.read_csv(p) for p in lbl_paths}
empty_lbl_df = []
for t, lbl_df in lbls.items():
    try:
        lbl_df.loc[:, 'traj'] = t
    except ValueError:
        empty_lbl_df.append(t)
print(f"Empty dfs for {empty_lbl_df}")
lbls = pd.concat(lbls.values(), axis=0)
lbls.loc[:, 'area'] = [t.split('_')[0] for t in lbls['traj']]

# Remove specific traj
# traj_to_remove = ['pat01_vc_re_run1', 'pat04_vc_le_run1', 'pat04_vc_le_run3', 'pat04_vc_re_run2', 'pat06_vc_re_run1', 'pat07_vc_re_run1', 'pat15_vc_le_run1']
# print(f"Removing {len(traj_to_remove)} of {len(lbls['traj'].unique())} trajectories")
# lbls = lbls[~lbls['traj'].isin(traj_to_remove)].copy()
# print(f"{len(lbls['traj'].unique())} remaining.")
lbls = lbls.groupby(['area', 'traj']).agg({'Label': lambda x: ', '.join(x.unique())}).reset_index()
lbls

In [ ]:
# Double check to make sure labels match
lbls = pd.merge(cli_dfs, lbls, left_on=['area', 'trajectory'], right_on=['area', 'traj']).drop(columns=['traj']).rename(columns={'Label': 'label_og'})
lbls[lbls['label'] != lbls['label_og']]

### Img count

In [ ]:
print(f"{len(cli_dfs['area'].unique())} patients")
print(f"{len(cli_dfs)} images")
# cli_dfs.groupby(['area']).agg(img_count=('img_relative_path', 'count')).reset_index()
cli_dfs.groupby(['label']).agg(img_count=('img_relative_path', 'count'), pat_count=('area', 'nunique')).reset_index()

In [ ]:
cli_dfs.groupby('subset').agg(img_count=('img_relative_path', 'nunique'), pat_count=('area', 'nunique'))

In [ ]:
cli_dfs.groupby(['subset', 'label']).agg(img_count=('img_relative_path', 'nunique'))

In [ ]:
cli_dfs.groupby(['area', 'trajectory', 'label']).agg(img_count=('img_relative_path', 'count')).reset_index()

In [ ]:
cli_dfs.groupby(['area', 'trajectory']).agg(img_count=('img_relative_path', 'count')).reset_index()
# pd.merge(cli_dfs.groupby(['label', 'area', 'trajectory']).agg(img_count=('img_relative_path', 'count')).reset_index(), cli_dfs[['label', 'area']].drop_duplicates(subset=['label', 'area']))

In [ ]:
split0 = {'train': ['pat02', 'pat04', 'pat10', 'pat16'], 'valid': ['pat07', 'pat09', 'pat11'], 'test': ['pat01', 'pat08', 'pat13', 'pat15']}
split1 = {'train': ['pat10', 'pat09', 'pat11', 'pat01'], 'valid': ['pat04', 'pat13', 'pat16'], 'test': ['pat08', 'pat02', 'pat07', 'pat15']}
split2 = {'train': ['pat10', 'pat11', 'pat16', 'pat07'], 'valid': ['pat04', 'pat15', 'pat01'], 'test': ['pat09', 'pat08', 'pat02', 'pat13']}
splits = {'split 0': split0, 
          'split 1': split1,
          'split 2': split2}
for s, split in splits.items():
    cli_df_split = cli_dfs.copy()
    # Reassig subset
    cli_df_split['subset'] = ''
    for subset, pats in split.items():
        cli_df_split.loc[cli_df_split['area'].isin(pats), 'subset'] = subset
    # Get stats
    subset_stats = cli_df_split.groupby('subset').agg({'img_relative_path':'count', 'area': lambda x: ', '.join(set(x))}).rename(columns={'img_relative_path': 'img_count', 'area': 'pats'}).reset_index()
    print(s)
    print(subset_stats)

## Augmentations

In [ ]:
# Define transforms
img_transforms = [# v2.ToTensor(),
                  v2.ToImage(),
                  v2.ToDtype(torch.float32, scale=True),
                  v2.Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
                  NormTransform(),
                  transforms.Grayscale()
    ]
train_aug = [
        transforms.RandomHorizontalFlip(p=1),
        transforms.RandomVerticalFlip(p=1),
        transforms.RandomEqualize(p=1),
        RandomWrapAround(dim=-1, p=1.0),
        RandomWrapAround(dim=-2, p=1.0)
    ]
train_aug_cli = [
                v2.RandomApply([v2.ColorJitter(brightness=0.25, contrast=0.25)], p=1),
                v2.RandomResizedCrop(size=256, scale=(0.75, 1), interpolation=InterpolationMode.BILINEAR),
                # https://arxiv.org/abs/2409.13351v1
                v2.RandomApply([v2.RandomAffine(degrees=5, interpolation=InterpolationMode.BILINEAR)], p=1),
                # v2.RandomApply([v2.RandomAffine(degrees=5, shear=5, interpolation=InterpolationMode.BILINEAR)], p=0.5),
                # v2.RandomApply([v2.ElasticTransform(alpha=100, sigma=[1e-5, 10])], p=0.5), # Long elastic
                # v2.RandomApply([v2.ElasticTransform(alpha=20, sigma=[1e-5, 2])], p=0.5),  # Short elastic
                # v2.RandomApply([v2.GaussianBlur(kernel_size=5)], p=0.5),
                # v2.RandomApply([v2.GaussianNoise(mean=0, sigma=0.1, clip=True)], p=0.5), # https://doi.org/10.1002/eng2.70110
                v2.RandomAutocontrast(p=1),
            ]

In [ ]:
# Set image paths
oct_imgs = {'chicken_heart_muscle': pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images\chicken_heart_p1_area1_sine_5x5y_s8_zamp05_3073_3584.png"),
            'lamb_heart_fat': pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images\lamb_heart_area1_sine_5x5y_s8_zamp05_54273_54784.png"),
            'lamb_heart_muscle': pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images\lamb_heart_area6_sine_5x5y_s8_zamp05_38401_38912.png"),
            'lamb_liver': pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images\lamb_liver_p1_area9_sine_5x5y_s8_zamp05_1025_1536.png"),
            'lamb_testicle': pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images\lamb_testicle_area3_sine_5x5y_s8_zamp05_43521_44032.png"),
            'pat_healthy': pathlib.Path(r"Y:\OCT_clinical_data\512mscans_sample20\pat01\pat01_vc_re_run2_1702281744505670_4097_4608.png"),
            'pat_lesion': pathlib.Path(r"Y:\OCT_clinical_data\512mscans_sample20\pat02\pat02_vc_re_run1_1706697962994786_35329_35840.png")}
# Choose class
lbl = list(oct_imgs.keys())[0]

# Load data and set to grayscale
img = Image.fromarray(cv2.imread(oct_imgs[lbl]))
# Convert to grayscale
img = img_transforms[-1](img)

In [ ]:

# Plot transforms
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
axs[0, 0].imshow(img, cmap='gray')
axs[0, 0].set_title('Original image')
axs[0, 1].imshow(train_aug[0](img), cmap='gray')
axs[0, 1].set_title('Horizontal Flip')
axs[0, 2].imshow(train_aug[1](img), cmap='gray')
axs[0, 2].set_title('Vertical Flip')
axs[1, 0].imshow(train_aug[2](img), cmap='gray')
axs[1, 0].set_title('Histogram Equalize')
axs[1, 1].imshow(train_aug[3](torch.tensor(np.array(img))), cmap='gray')
axs[1, 1].set_title('Random Wrap Around (horizontal)')
axs[1, 2].imshow(train_aug[4](torch.tensor(np.array(img))), cmap='gray')
axs[1, 2].set_title('Random Wrap Around (vertical)')
for ax in axs.ravel():
    ax.set_axis_off()
# plt.suptitle(f'Transforms applied on the images in the training dataset')
plt.tight_layout()
# plt.savefig(f'train_aug_{lbl}.png', dpi=500)
plt.show()

In [ ]:
# Plot transforms
fig, axs = plt.subplots(2, 2, figsize=(8, 6))
axs[0, 0].imshow(train_aug_cli[0](img), cmap='gray')
axs[0, 0].set_title('Color Jitter')
axs[0, 1].imshow(train_aug_cli[1](img), cmap='gray')
axs[0, 1].set_title('Random Resized Crop')
axs[1, 0].imshow(train_aug_cli[2](img), cmap='gray')
axs[1, 0].set_title('Random Rotation')
axs[1, 1].imshow(train_aug_cli[3](img), cmap='gray')
axs[1, 1].set_title('Random Autocontrast')
for ax in axs.ravel():
    ax.set_axis_off()
# plt.suptitle(f'Transforms applied on the training images of the clinical dataset')
plt.tight_layout()
plt.savefig(f'train_aug_cli_{lbl}.png', dpi=500)
plt.show()

In [ ]:
# Plot sample for SSL input with iip
img_root_path = pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\figures\oct_images")
oct_imgs = {'chicken_heart_muscle': [img_root_path.joinpath("chicken_heart_p1_area2_sine_5x5y_s8_zamp05_8705_9216.png"), img_root_path.joinpath("scans_same_area/chicken_heart_p1_area2_sine_5x5y_s8_zamp05_17921_18432.png")],
            'lamb_heart_fat': [img_root_path.joinpath("lamb_heart_area1_sine_5x5y_s8_zamp05_54273_54784.png"), img_root_path.joinpath("scans_same_area/lamb_heart_area1_sine_5x5y_s8_zamp05_28161_28672.png")],
            'lamb_heart_muscle': [img_root_path.joinpath("lamb_heart_area6_sine_5x5y_s8_zamp05_38401_38912.png"), img_root_path.joinpath("scans_same_area/lamb_heart_area6_sine_5x5y_s8_zamp05_40449_40960.png")],
            'lamb_liver': [img_root_path.joinpath("lamb_liver_p1_area9_sine_5x5y_s8_zamp05_1025_1536.png"), img_root_path.joinpath("scans_same_area/lamb_liver_p1_area9_sine_5x5y_s8_zamp05_35329_35840.png")],
            'lamb_testicle': [img_root_path.joinpath("lamb_testicle_area3_sine_5x5y_s8_zamp05_43521_44032.png"), img_root_path.joinpath("scans_same_area/lamb_testicle_area3_sine_5x5y_s8_zamp05_28673_29184.png")],
            }
# Choose class
lbl = list(oct_imgs.keys())[0]

# Define transforms
img_transforms = [# v2.ToTensor(),
                  v2.ToImage(),
                  v2.ToDtype(torch.float32, scale=True),
                  v2.Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
                  NormTransform(),
                  transforms.Grayscale()
]
aug = [
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomWrapAround(dim=-1, p=1.0),
    RandomWrapAround(dim=-2, p=1.0)
]
ssl_transforms = [transforms.RandomEqualize(p=0.5),] + img_transforms + aug
ssl_transforms = transforms.Compose(ssl_transforms)


In [ ]:
# Load data and set to grayscale
for img_path in oct_imgs[lbl]:
    img = Image.fromarray(cv2.imread(img_path))
    # Apply transform
    img_t = ssl_transforms(img)
    img_t = img_t[0, :, :]
    plt.imshow(img_t, cmap='gray')
    plt.axis('off')
    # fig.savefig('out.png', bbox_inches='tight', pad_inches=0)
    plt.savefig(img_root_path.joinpath(f"ssl_trans_{img_path.stem}.png"), bbox_inches='tight', pad_inches=0, dpi=500)
    plt.show()

## Param count

In [ ]:
# ResNet
rn18 = models.resnet18()
sum(p.numel() for p in rn18.parameters()) - sum(p.numel() for p in rn18.fc.parameters()) 

In [ ]:
# PVT
pvt = timm.create_model("pvt_v2_b0", pretrained=True)
sum(p.numel() for p in pvt.parameters()) - sum(p.numel() for p in pvt.head.parameters()) 

In [ ]:
# Mobilenet
mn = models.mobilenet_v3_small()
sum(p.numel() for p in mn.parameters()) - sum(p.numel() for p in mn.classifier.parameters()) 

## Load predictions

In [ ]:
def find_pred_files(folder_path: pathlib.Path, dataset_name:str, keep_cv=False) -> list:
    files = list(folder_path.glob('*.csv'))
    if dataset_name == 'oct':
        files = [p for p in files if 'clinical' not in p.__str__()]
    else:
        files = [p for p in files if 'clinical' in p.__str__()]
        if not keep_cv:
            files = [p for p in files if 'split' not in p.__str__()]
    return files

def merge_pred_csv_V0(pred_paths: list)-> pd.DataFrame:
    preds = {p: pd.read_csv(p) for p in pred_paths}
    preds_all = pd.DataFrame()
    for p, df in preds.items():
        if ('random' in p.stem) or ('default' in p.stem):
            starting_weights = p.stem.split('_')[-1]
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-2]))
        else:
            starting_weights = p.parent.parent.stem
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-1]))
        col_name = f"{starting_weights}_{re.sub(r'[^\w\s]', '', f'{ratio_sup:03d}')}"
        # print(col_name)
        df = df.rename(columns={'pred': f'pred_{col_name}'})
        df = df.set_index([c for c in df.columns if 'pred' not in c])
        if len(preds_all) == 0:
            preds_all = df.copy()
        else:
            preds_all = pd.merge(preds_all, df, left_index=True, right_index=True, how='outer')
    preds_all = preds_all.reindex(sorted(preds_all.columns), axis=1)
    return preds_all

def merge_pred_csv(pred_paths:list)->pd.DataFrame:
    preds = {p: pd.read_csv(p) for p in pred_paths}
    preds_all = pd.DataFrame()
    for p, df in preds.items():
        # Add random outputs to df if none are saved ()
        if len([c for c in df.columns if 'prob' in c]) == 0:
            out_df = np.random.rand(len(df), len(df['label'].unique()))
            out_df = pd.DataFrame(out_df, columns=[f'prob_{int(i)}' for i in range(len(df['label'].unique()))])
            out_df = pd.concat([out_df, df['pred']], axis=1)
            df = df.drop(columns=['pred'])
            df = pd.concat([df, out_df], axis=1)
        # Return to normal function
        if 'split' in p.stem:
            p = p.parent.joinpath(f'{'_'.join(p.stem.split('_')[:-1])}{p.suffix}')
        if 'noLoRA' in p.stem:
            p = p.parent.joinpath(f'{'_'.join(p.stem.split('_')[:-1])}{p.suffix}')
        if ('random' in p.stem) or ('default' in p.stem):
            starting_weights = p.stem.split('_')[-1]
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-2]))
        else:
            starting_weights = p.parent.parent.stem
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-1]))
        col_name = f"{starting_weights}_{re.sub(r'[^\w\s]', '', f'{ratio_sup:03d}')}"
        # print(col_name)
        new_col_names = {c: f'{c}_{col_name}' for c in df.columns if 'prob' in c}
        new_col_names.update({'pred': f'pred_{col_name}'})
        df = df.rename(columns=new_col_names)
        # df = df.set_index([c for c in df.columns if 'pred|prob' not in c])
        df = df.set_index([c for c in df.columns if not re.search('pred|prob', c)])
        if len(preds_all) == 0:
            preds_all = df.copy()
        else:
            preds_all = pd.merge(preds_all, df, left_index=True, right_index=True, how='outer')
    preds_all = preds_all.reindex(sorted(preds_all.columns), axis=1)
    return preds_all

def merge_pred_csv_clinical_cv(pred_paths:list)->dict:
    pred_paths = {p.stem.split('_')[-1]: [p] for p in pred_paths}
    preds_per_split = {}
    for s, p in pred_paths.items():
        preds_per_split[s] = merge_pred_csv(p)
    return preds_per_split

def merge_pred_csv_clinical(pred_paths:list)->dict:
    pred_paths = {p.stem.split('_')[-1]: [p] for p in pred_paths}
    preds_per_split = {}
    for s, p in pred_paths.items():
        preds_per_split[s] = merge_pred_csv(p)
    return preds_per_split

def load_preds(pred_paths:list, dataset:str):
    if dataset == 'oct':
        df =  merge_pred_csv(pred_paths)
    elif dataset == 'oct_clinical':
        df =  merge_pred_csv(pred_paths)
    return df

def get_f1_score_per_col(df: pd.DataFrame, keep_weight_init=True) -> dict:
    pred_cols = [c for c in df.columns if 'pred' in c]
    df = df.reset_index()
    f1_dict = {}
    for c in pred_cols:
        df_c = df.dropna(subset=[c])
        # f1s_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average=None) # F1 score per class
        f1_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average='macro') # Mean of F1 scores per class
        if keep_weight_init:
            f1_dict[c] = f1_c
        else:
            new_c = c.split('_')[-1]
            f1_dict[new_c] = f1_c
    return f1_dict

def get_map_per_col(df: pd.DataFrame, keep_weight_init=True) -> dict:
    prob_cols = [c for c in df.columns if 'prob' in c]
    ratios = list(set([pc.split('_')[-1] for pc in prob_cols]))
    ratios.sort()
    df = df.reset_index()
    map_dict = {}
    for r in ratios:
        r_cols = [c for c in prob_cols if r in c]
        df_r = df.dropna(subset=r_cols)
        map_r = average_precision_score(df_r['label'].to_numpy(), df_r[r_cols].to_numpy(), average='macro') # Mean of F1 scores per class
        if keep_weight_init:
            new_r = '_'.join([s for i, s in enumerate(r_cols[0].split('_')) if i != 1])
            map_dict[new_r] = map_r
        else:
            new_r = r.split('_')[-1]
            map_dict[new_r] = map_r
    return map_dict

def get_f1_score_per_col(df: pd.DataFrame, keep_weight_init=True) -> dict:
    pred_cols = [c for c in df.columns if 'pred' in c]
    df = df.reset_index()
    report_dict = {}
    for c in pred_cols:
        df_c = df.dropna(subset=[c])
        # f1s_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average=None) # F1 score per class
        # f1_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average='macro') # Mean of F1 scores per class
        report_c = classification_report(df_c['label'].tolist(), df_c[c].tolist(), 
                                         target_names=df_c['label_str'].unique().tolist(), 
                                         output_dict=True,
                                         digits=4, zero_division=np.nan)
        if keep_weight_init:
            report_dict[c] = report_c
        else:
            new_c = c.split('_')[-1]
            report_dict[new_c] = report_c
    return report_dict

In [ ]:
model_archs = {'resnet18': 'ResNet-18', 'pvtv2b0': 'PVTv2-b0', 'mobilenetv3': 'MobileNetV3'}
dino_archs = {'vits': 'ViT-S', 'vitb': 'ViT-B', 'vitl': 'ViT-L'}
model_arch = list(model_archs.keys())[1]
dino_arch = list(dino_archs.keys())[2]
datasets = {'oct': 'lab', 'oct_clinical': 'clinical'}
dataset = list(datasets.keys())[1]

In [ ]:
# Load all pred files
# MAKE SURE FILES HAVE BEEN DOWNLOADED FROM THE HPC
# Define base folder paths
supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{model_arch}')
byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{model_arch}')
simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{model_arch}')
dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{dino_arch}')

# Find pred files
supervised_pred_files = find_pred_files(supervised_folder, dataset)
byol_pred_files = find_pred_files(byol_folder, dataset)
simclr_pred_files = find_pred_files(simclr_folder, dataset)
dino_pred_files = find_pred_files(dino_folder, dataset)

# Split random vs imagenet preds
supervised_pred_random_files = [p for p in supervised_pred_files if 'random' in p.__str__()]
supervised_pred_imagenet_files = [p for p in supervised_pred_files if 'default' in p.__str__()]
# Split LoRA and noLoRA preds
dino_pred_lora_files = [p for p in dino_pred_files if not 'noLoRA' in p.__str__()]
dino_pred_noLora_files = [p for p in dino_pred_files if 'noLoRA' in p.__str__()]

# Load preds as csv
supervised_preds_random = load_preds(supervised_pred_random_files, dataset)
supervised_preds_imagenet = load_preds(supervised_pred_imagenet_files, dataset)
byol_preds = load_preds(byol_pred_files, dataset)
simclr_preds = load_preds(simclr_pred_files, dataset)
dino_preds_lora = load_preds(dino_pred_lora_files, dataset)
dino_preds_noLora = load_preds(dino_pred_noLora_files, dataset)


## Plot experiment 1 results

### SSL plots - lab

In [ ]:
# Get mAP per col
all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
map_overview = {}
for w, preds in all_preds.items():
    map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
map_overview.index = [int(i) for i in map_overview.index]
map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
print(map_overview)
# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
map_overview.plot.line(x='train_ratio', y=[c for c in map_overview if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      marker='o')
plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
plt.show()


In [ ]:
# Get F1 per col
all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
f1_overview = {}
for w, preds in all_preds.items():
    f1_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
f1_overview = pd.DataFrame.from_dict(f1_overview, orient='columns')
f1_overview.index = [int(i) for i in f1_overview.index]
f1_overview = f1_overview.reset_index().rename(columns={'index': 'train_ratio'})
print(f1_overview)
# f1_overview
plt_title = f"F1 score on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
f1_overview.plot.line(x='train_ratio', y=[c for c in f1_overview if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro F1 score',
                      marker='o')
# plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)

### DINO plots - lab

In [ ]:
# Get mAP per col
all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
map_overview = {}
for w, preds in all_preds.items():
    map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
map_overview.index = [int(i) for i in map_overview.index]
map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
print(map_overview)
# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset using a {dino_archs[dino_arch]} encoder"
map_overview.plot.line(x='train_ratio', y=[c for c in map_overview if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      marker='o')
plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
plt.show()

In [ ]:
# Get F1 per col
all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
f1_overview = {}
for w, preds in all_preds.items():
    f1_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
f1_overview = pd.DataFrame.from_dict(f1_overview, orient='columns')
f1_overview.index = [int(i) for i in f1_overview.index]
f1_overview = f1_overview.reset_index().rename(columns={'index': 'train_ratio'})
print(f1_overview)
# f1_overview
plt_title = f"F1 score on the {datasets[dataset]} dataset using a {dino_archs[dino_arch]} encoder"
f1_overview.plot.line(x='train_ratio', y=[c for c in f1_overview if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro F1 score',
                      marker='o')
# plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)

### SSL - clinical

In [ ]:
# Get mAP per col
all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
split_list = list(set(sum([list(d.keys()) for d in list(all_preds.values())], [])))
split_list.sort()
map_overview = {}
for w, preds in all_preds.items():
    map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
# map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
map_overview.index = [int(i) for i in map_overview.index]
map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
print(map_overview)

# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
map_overview.drop(columns=['train_ratio']).T[0].plot.bar(title=plt_title,
                    xlabel=r'% of the training set',
                    ylabel='Macro mean average precision',)
plt.xticks(rotation=0, ha='center')
plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
plt.show()

map_ssl_oct_clinical = map_overview.copy()


In [ ]:
# Get F1 per col
all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
split_list = list(set(sum([list(d.keys()) for d in list(all_preds.values())], [])))
split_list.sort()
map_overview = {}
for s in split_list:
    map_overview[s] = {}
    split_preds = {k: df[s] for k, df in all_preds.items()}
    for w, preds in split_preds.items():
        map_overview[s][w] = get_f1_score_per_col(preds, keep_weight_init=False)['100']
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
print(map_overview)
# f1_overview
plt_title = f"F1 score on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
map_overview.plot.bar(x='split', y=[c for c in map_overview if c != 'split'], 
                    title=plt_title,
                    xlabel=r'% of the training set',
                    ylabel='Macro F1 score',
                    )
plt.xticks(rotation=0, ha='center')

### DINO - clinical

In [ ]:
# Get mAP per col
# dino_preds_noLora = dino_preds_lora # TODO: Remove once files have been generated
all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
split_list = list(set(sum([list(d.keys()) for d in list(all_preds.values())], [])))
split_list.sort()
map_overview = {}
for s in split_list:
    map_overview[s] = {}
    split_preds = {k: df[s] for k, df in all_preds.items()}
    for w, preds in split_preds.items():
        map_overview[s][w] = get_map_per_col(preds, keep_weight_init=False)['100']
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
print(map_overview)
# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset using a {dino_archs[dino_arch]} encoder"
map_overview.plot.bar(x='split', y=[c for c in map_overview if c != 'split'], 
                    title=plt_title,
                    xlabel=r'% of the training set',
                    ylabel='Macro mean average precision',
                    )
plt.xticks(rotation=0, ha='center')
plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
plt.show()


In [ ]:
# Get F1 per col
# dino_preds_noLora = dino_preds_lora # TODO: Remove once files have been generated
all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
split_list = list(set(sum([list(d.keys()) for d in list(all_preds.values())], [])))
split_list.sort()
map_overview = {}
for s in split_list:
    map_overview[s] = {}
    split_preds = {k: df[s] for k, df in all_preds.items()}
    for w, preds in split_preds.items():
        map_overview[s][w] = get_f1_score_per_col(preds, keep_weight_init=False)['100']
map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
print(map_overview)
# f1_overview
plt_title = f"F1 score on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
map_overview.plot.bar(x='split', y=[c for c in map_overview if c != 'split'], 
                    title=plt_title,
                    xlabel=r'% of the training set',
                    ylabel='Macro F1 score',
                    )
plt.xticks(rotation=0, ha='center')

## Experiment 2

### Lab dataset

In [ ]:
# Get mAP per col
dataset = 'oct'
map_per_arch = []
for arch in model_archs.keys():
    # Load all pred files
    # Define base folder paths
    supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{arch}')
    byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{arch}')
    simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{arch}')

    # Find pred files
    supervised_pred_files = find_pred_files(supervised_folder, dataset)
    byol_pred_files = find_pred_files(byol_folder, dataset)
    simclr_pred_files = find_pred_files(simclr_folder, dataset)

    # Split random vs imagenet preds
    supervised_pred_random_files = [p for p in supervised_pred_files if 'random' in p.__str__()]
    supervised_pred_imagenet_files = [p for p in supervised_pred_files if 'default' in p.__str__()]

    # Load preds as csv
    supervised_preds_random = load_preds(supervised_pred_random_files, dataset)
    supervised_preds_imagenet = load_preds(supervised_pred_imagenet_files, dataset)
    byol_preds = load_preds(byol_pred_files, dataset)
    simclr_preds = load_preds(simclr_pred_files, dataset)

    all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}

    map_overview = {}
    for w, preds in all_preds.items():
        map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
    map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
    map_overview.index = [int(i) for i in map_overview.index]
    map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
    # Save exp 1 figure
    # f1_overview
    plt_title = f"mAP on the {datasets[dataset]} dataset using a {model_archs[arch]} encoder"
    map_overview.plot.line(x='train_ratio', y=[c for c in map_overview if c != 'train_ratio'], 
                        title=plt_title,
                        xlabel=r'% of the training set',
                        ylabel='Macro mean average precision',
                        ylim=[0.1, 1],
                        marker='o',
                        # figsize=(8, 5)
                        )
    # plt.legend(bbox_to_anchor=(1.0, 0.0), loc='center right')
    plt.tight_layout()
    plt.savefig(f'F1score_{arch}_{dataset}.png', dpi=500)
    plt.close()
    map_overview['arch'] = arch
    # print(map_overview)
    map_per_arch.append(map_overview)
# Group per train ratio
map_per_arch = pd.concat(map_per_arch, axis=0)
map_per_arch_print = map_per_arch.pivot_table(index='train_ratio', columns=['arch'], values=['scratch', 'ImageNet', 'SSL-BYOL', 'SSL-SimCLR'])
map_per_arch_print.columns = map_per_arch_print.columns.swaplevel(0)
map_per_arch_print = map_per_arch_print.sort_index(level=0, axis=1)
# print(map_per_arch_print.T)
map_per_arch = map_per_arch.drop(columns=['arch']).groupby('train_ratio').mean().reset_index()

# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset averaged over all encoder architectures"
map_per_arch.plot.line(x='train_ratio', y=[c for c in map_per_arch if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      marker='o',
                      ylim=[0.1, 1],
                      figsize=(8, 5))
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.subplots_adjust(right=0.7)
plt.tight_layout()
plt.savefig(f'F1score_avg_{dataset}.png', dpi=500)
plt.show()

# Save results for other figure
map_ssl_oct = map_per_arch_print.T.copy()

In [ ]:
# Experiment 2 numerical results
map_per_arch

In [ ]:
# Experiment 1 numerical results
map_per_arch_print.T

In [ ]:
df = map_per_arch_print.T.copy()
df['diff_20_5'] = df[20] - df[5]
df['mean'] = df[[5, 10, 15, 20]].mean(axis=1)
df

### Clinical dataset

In [ ]:
# Get mAP per col
dataset = 'oct_clinical'
map_per_arch = []
for arch in model_archs.keys():
    # Load all pred files
    # Define base folder paths
    supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{arch}')
    byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{arch}')
    simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{arch}')

    # Find pred files
    supervised_pred_files = find_pred_files(supervised_folder, dataset)
    byol_pred_files = find_pred_files(byol_folder, dataset)
    simclr_pred_files = find_pred_files(simclr_folder, dataset)

    # Split random vs imagenet preds
    supervised_pred_random_files = [p for p in supervised_pred_files if 'random' in p.__str__()]
    supervised_pred_imagenet_files = [p for p in supervised_pred_files if 'default' in p.__str__()]

    # Load preds as csv
    supervised_preds_random = load_preds(supervised_pred_random_files, dataset)
    supervised_preds_imagenet = load_preds(supervised_pred_imagenet_files, dataset)
    byol_preds = load_preds(byol_pred_files, dataset)
    simclr_preds = load_preds(simclr_pred_files, dataset)

    # Get mAP per col
    all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
    map_overview = {}
    for w, preds in all_preds.items():
        map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
    map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
    # map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
    map_overview.index = [int(i) for i in map_overview.index]
    map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
    # print(map_overview)

    # f1_overview
    plt_title = f"mAP on the {datasets[dataset]} dataset using a {model_archs[arch]} encoder"
    map_overview.drop(columns=['train_ratio']).T[0].plot.bar(title=plt_title,
                        # xlabel=r'% of the training set',
                        ylabel='Macro mean average precision',
                        ylim=[0.1, 1],)
    plt.xticks(rotation=0, ha='center')
    plt.savefig(f'F1score_{arch}_{dataset}.png', dpi=500)
    plt.close()
    map_overview['arch'] = arch
    # print(map_overview)
    map_per_arch.append(map_overview)

# Group per train ratio
map_per_arch = pd.concat(map_per_arch, axis=0)
print(map_per_arch)
# map_per_arch_print = map_per_arch.pivot_table(index='split', columns=['arch'], values=['scratch', 'ImageNet', 'SSL-BYOL', 'SSL-SimCLR'])
# map_per_arch_print.columns = map_per_arch_print.columns.swaplevel(0)
# map_per_arch_print = map_per_arch_print.sort_index(level=0, axis=1)
# print(map_per_arch_print.T)
# map_per_arch = map_per_arch.drop(columns=['arch']).groupby('split').mean().reset_index()

# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset averaged over all encoder architectures"
map_per_arch_plt = map_per_arch.drop(columns=['train_ratio','arch']).mean()
map_per_arch_plt.plot.bar( 
                    title=plt_title,
                    # xlabel=r'% of the training set',
                    ylabel='Macro mean average precision',
                    ylim=[0.1, 1],
                    figsize=(8, 5))
plt.xticks(rotation=0, ha='center')
plt.tight_layout()
plt.savefig(f'F1score_avg_{dataset}.png', dpi=500)
plt.show()

map_ssl_oct_clinical = map_per_arch.copy()


In [ ]:
# Experiment 2 numerical results
map_per_arch_plt

In [ ]:
# Experiment 1 numerical results
map_per_arch

In [ ]:
# Merged exp1 figure
plt_title = f"mAP on the clinical dataset per encoder architecture"
# map_per_arch['arch'] = [model_archs[a] for a in map_per_arch['arch']]
map_per_arch.drop(columns=['train_ratio']).set_index('arch').plot.bar( 
                    title=plt_title,
                    xlabel=r'',
                    ylabel='Macro mean average precision',
                    ylim=[0.1, 1],
                    figsize=(8, 5))
plt.xticks(rotation=0, ha='center')
plt.tight_layout()
plt.savefig(f'F1score_ssl_all_clinical.png', dpi=500)
plt.show()

In [ ]:
map_per_arch

In [ ]:
map_per_arch[['SSL-BYOL', 'SSL-SimCLR']].mean(axis=1) - map_per_arch['ImageNet']

### DINO - lab (generate all exp1 figures)

In [ ]:
# Get mAP per col
dataset = 'oct'
map_per_arch = []
for arch in dino_archs.keys():
    # Load all pred files
    # Define base folder paths
    dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{arch}')

    # Find pred files
    dino_pred_files = find_pred_files(dino_folder, dataset)
    # Split LoRA and noLoRA preds
    dino_pred_lora_files = [p for p in dino_pred_files if not 'noLoRA' in p.__str__()]
    dino_pred_noLora_files = [p for p in dino_pred_files if 'noLoRA' in p.__str__()]

    # Load preds as csv
    dino_preds_lora = load_preds(dino_pred_lora_files, dataset)
    dino_preds_noLora = load_preds(dino_pred_noLora_files, dataset)

    # Get mAP per col
    all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
    map_overview = {}
    for w, preds in all_preds.items():
        map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
    map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
    map_overview.index = [int(i) for i in map_overview.index]
    map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})

    # f1_overview
    plt_title = f"mAP on the {datasets[dataset]} dataset using a {dino_archs[arch]} encoder"
    map_overview.plot.line(x='train_ratio', y=[c for c in map_overview if c != 'train_ratio'], 
                        title=plt_title,
                        xlabel=r'% of the training set',
                        ylabel='Macro mean average precision',
                        ylim=[0.1, 1],
                        marker='o')
    plt.tight_layout()
    plt.savefig(f'F1score_{arch}_{dataset}.png', dpi=500)
    # plt.show()
    plt.close()
    map_overview['arch'] = arch
    # print(map_overview)
    map_per_arch.append(map_overview)

# Group per train ratio
map_per_arch = pd.concat(map_per_arch, axis=0)
map_per_arch_print = map_per_arch.pivot_table(index='train_ratio', columns=['arch'], values=['DINOv3', 'DINOv3 with LoRA'])
map_per_arch_print.columns = map_per_arch_print.columns.swaplevel(0)
map_per_arch_print = map_per_arch_print.sort_index(level=0, axis=1)
# print(map_per_arch_print.T)
map_per_arch = map_per_arch.drop(columns=['arch']).groupby('train_ratio').mean().reset_index()

# f1_overview
plt_title = f"mAP on the {datasets[dataset]} dataset averaged over all encoder architectures"
map_per_arch.plot.line(x='train_ratio', y=[c for c in map_per_arch if c != 'train_ratio'], 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      marker='o',
                      ylim=[0.1, 1],
                      figsize=(8, 5))
plt.tight_layout()
plt.savefig(f'F1score_dino_avg_{dataset}.png', dpi=500)
plt.show()

map_dino_oct = map_per_arch_print.T.copy()


In [ ]:
# Experiment 2 numerical results
map_per_arch

In [ ]:
# Experiment 1 numerical results
map_per_arch_print.T

In [ ]:
df = map_per_arch_print.T.reset_index().rename(columns={'level_1': 'model'})
df['arch'] = [f"ViT-{df.loc[i, 'arch'][-1].capitalize()}{' with LoRA' if 'LoRA' in df.loc[i, 'model'] else ''}" for i in df.index]
df = df.set_index('arch').reindex(['ViT-S', 'ViT-S with LoRA', 'ViT-B', 'ViT-B with LoRA', 'ViT-L', 'ViT-L with LoRA']).reset_index()
#df = df.drop(columns=['model'])
df = df.rename(columns={'arch': 'Model'})
df

# Merged plot
fig, ax = plt.subplots(1, 1)
plt_title = f"mAP on the lab dataset per DINOv3 encoder architecture"
# Plot no lora
df[df['model'] == 'DINOv3'].set_index('Model').drop(columns=['model']).T.plot.line(
    ax=ax,
    title=plt_title,
    xlabel=r'% of the training set',
    ylabel='Macro mean average precision',
    style=['o-r', 'o-g', 'o-b'],
    # marker='o',
    ylim=[0.1, 1],
    figsize=(8, 5))
# Plot with lora
df[df['model'] == 'DINOv3 with LoRA'].set_index('Model').drop(columns=['model']).T.plot.line(
    ax=ax,
    title=plt_title,
    xlabel=r'% of the training set',
    ylabel='Macro mean average precision',
    style=['s-r', 's-g', 's-b'],
    # marker='s',
    ylim=[0.1, 1],
    figsize=(8, 5))
plt.tight_layout()
plt.savefig(f'F1score_dino_all_oct.png', dpi=500)
plt.show()


### DINO - clinical (generate all exp1 figures)

In [ ]:
# Get mAP per col
dataset = 'oct_clinical'
map_per_arch = []
for arch in dino_archs.keys():
    # Load all pred files
    # Define base folder paths
    dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{arch}')

    # Find pred files
    dino_pred_files = find_pred_files(dino_folder, dataset)
    # print(f"{arch}: {len(dino_pred_files)}")
    # print(dino_pred_files)
    # Split LoRA and noLoRA preds
    dino_pred_lora_files = [p for p in dino_pred_files if not 'noLoRA' in p.__str__()]
    dino_pred_noLora_files = [p for p in dino_pred_files if 'noLoRA' in p.__str__()]
    # print(f"{arch}: {len(dino_pred_noLora_files)}")
    # Load preds as csv
    dino_preds_lora = load_preds(dino_pred_lora_files, dataset)
    dino_preds_noLora = load_preds(dino_pred_noLora_files, dataset)
    # print(f"{arch}: {len(dino_preds_noLora)}")

    # Get mAP per col
    all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
    map_overview = {}
    for w, preds in all_preds.items():
        map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
    map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
    # map_overview = map_overview.T.reset_index().rename(columns={'index': 'split'})
    map_overview.index = [int(i) for i in map_overview.index]
    map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
    # print(map_overview)

    # f1_overview
    plt_title = f"mAP on the {datasets[dataset]} dataset using a {dino_archs[arch]} encoder"
    map_overview.drop(columns=['train_ratio']).T[0].plot.bar(title=plt_title,
                        # xlabel=r'% of the training set',
                        ylabel='Macro mean average precision',
                        ylim=[0.1, 1],)
    plt.xticks(rotation=0, ha='center')
    plt.savefig(f'F1score_{arch}_{dataset}.png', dpi=500)
    plt.close()
    map_overview['arch'] = arch
    # print(map_overview)
    map_per_arch.append(map_overview)

# Group per train ratio
map_per_arch = pd.concat(map_per_arch, axis=0)
print(map_per_arch)
# map_per_arch_print = map_per_arch.pivot_table(index='split', columns=['arch'], values=['DINOv3', 'DINOv3 with LoRA'])
# map_per_arch_print.columns = map_per_arch_print.columns.swaplevel(0)
# map_per_arch_print = map_per_arch_print.sort_index(level=0, axis=1)
# print(map_per_arch_print.T)
# map_per_arch = map_per_arch.drop(columns=['arch']).groupby('split').mean().reset_index()

plt_title = f"mAP on the {datasets[dataset]} dataset averaged over all encoder architectures"
map_per_arch_plt = map_per_arch.drop(columns=['train_ratio','arch']).mean()
map_per_arch_plt.plot.bar( 
                    title=plt_title,
                    # xlabel=r'% of the training set',
                    ylabel='Macro mean average precision',
                    ylim=[0.1, 1],
                    figsize=(8, 5))
plt.xticks(rotation=0, ha='center')
plt.tight_layout()
plt.savefig(f'F1score_dino_avg_{dataset}.png', dpi=500)
plt.show()

map_dino_oct_clinical = map_per_arch.copy()


In [ ]:
# Experiment 2 numerical results
map_per_arch_plt

In [ ]:
# Experiment 1 numerical results
map_per_arch

In [ ]:
df = map_per_arch.drop(columns=['train_ratio']).reset_index(drop=True).copy()
df['arch'] = [f"ViT-{df.loc[i, 'arch'][-1].capitalize()}" for i in df.index]
df = df.rename(columns={'arch': 'Model'})
df

# Merged plot
plt_title = f"mAP on the clinical dataset per DINOv3 encoder architecture"
df.set_index('Model').plot.bar(
    title=plt_title,
    # xlabel=r'% of the training set',
    ylabel='Macro mean average precision',
    ylim=[0.1, 1],
    figsize=(8, 5))
plt.xticks(rotation=0, ha='center')
plt.tight_layout()
plt.savefig(f'F1score_dino_all_oct_clinical.png', dpi=500)
plt.show()

## New exp2 with both SSL and DINO

### Lab dataset
Goal: Plot, per train_ratio, the best result over all encoder architectures

In [ ]:
map_ssl_oct_plt = map_ssl_oct.reset_index().rename(columns={'level_1': 'weights'}).drop(columns=['arch']).groupby('weights').max()
map_dino_oct_plt = map_dino_oct.reset_index().rename(columns={'level_1': 'weights'}).drop(columns=['arch']).groupby('weights').max()
map_overall_oct = pd.concat([map_ssl_oct_plt, map_dino_oct_plt], axis=0)
map_overall_oct = map_overall_oct.reindex(['scratch', 'ImageNet', 'SSL-BYOL', 'SSL-SimCLR', 'DINOv3', 'DINOv3 with LoRA'])
map_overall_oct

# f1_overview
plt_title = f"Max mAP on the lab dataset over all encoder architectures"
map_overall_oct.T.plot.line( 
                      title=plt_title,
                      xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      style=['o-', 'o-', 'o-', 'o-', 's-', 's-'],
                      # marker='o',
                      ylim=[0.1, 1],
                      figsize=(8, 5))
plt.tight_layout()
plt.savefig(f'F1score_max_oct.png', dpi=500)
plt.show()


In [ ]:
map_overall_oct_arch = []
for r in map_overall_oct.columns:
    df = map_overall_oct[r].to_frame().rename(columns={r: 'mAP'})
    df['train_ratio'] = r
    map_overall_oct_arch.append(df)
map_overall_oct_arch = pd.concat(map_overall_oct_arch, axis=0).reset_index()
map_overall_oct_arch['arch'] = ''

# Find corresponding architecture
ssl_match = map_ssl_oct.reset_index().rename(columns={'level_1': 'weights'})
dino_match = map_dino_oct.reset_index().rename(columns={'level_1': 'weights'})
for i in range(len(map_overall_oct_arch)):
    weights = map_overall_oct_arch.loc[i, 'weights']
    train_ratio = map_overall_oct_arch.loc[i, 'train_ratio']
    map_match = map_overall_oct_arch.loc[i, 'mAP']
    if 'DINO' in weights:
        arch_match = dino_match.loc[(dino_match['weights'] == weights) & (dino_match[train_ratio] == map_match),'arch'].iloc[0]
    else:
        arch_match = ssl_match.loc[(ssl_match['weights'] == weights) & (ssl_match[train_ratio] == map_match),'arch'].iloc[0]
    map_overall_oct_arch.loc[i, 'arch'] = arch_match
map_overall_oct_arch = map_overall_oct_arch.set_index(['weights', 'train_ratio'])
map_overall_oct_arch.sort_index()

In [ ]:
map_overall_oct_arch.loc[~map_overall_oct_arch.index.get_level_values(0).str.contains('DINO'), :]

### Clinical dataset
Goal: Plot the best result over all encoder architectures

In [ ]:
map_ssl_oct_cli_plt = map_ssl_oct_clinical.drop(columns=['train_ratio', 'arch']).max()
map_dino_oct_cli_plt = map_dino_oct_clinical.drop(columns=['train_ratio', 'arch']).max()
map_overall_oct_clinical = pd.concat([map_ssl_oct_cli_plt, map_dino_oct_cli_plt], axis=0)
map_overall_oct_clinical

# f1_overview
plt_title = f"Max mAP on the clinical dataset over all encoder architectures"
map_overall_oct_clinical.plot.bar( 
                      title=plt_title,
                      # xlabel=r'% of the training set',
                      ylabel='Macro mean average precision',
                      color=['b', 'b', 'b', 'b', 'r', 'r'],
                      ylim=[0.1, 1],
                      figsize=(8, 5))

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'F1score_max_oct_clinical.png', dpi=500)
plt.show()

In [ ]:
map_overall_oct_clinical_arch = map_overall_oct_clinical.to_frame().reset_index().rename(columns={'index': 'weights', 0: 'mAP'})
map_overall_oct_clinical_arch['arch'] = ''

# Find corresponding architecture
ssl_match = map_ssl_oct_clinical
dino_match = map_dino_oct_clinical
for i in range(len(map_overall_oct_clinical_arch)):
    weights = map_overall_oct_clinical_arch.loc[i, 'weights']
    map_match = map_overall_oct_clinical_arch.loc[i, 'mAP']
    if 'DINO' in weights:
        arch_match = dino_match.loc[(dino_match[weights] == map_match),'arch'].iloc[0]
    else:
        arch_match = ssl_match.loc[(ssl_match[weights] == map_match),'arch'].iloc[0]
    map_overall_oct_clinical_arch.loc[i, 'arch'] = arch_match
# map_overall_oct_arch = map_overall_oct_arch.set_index(['weights'])
map_overall_oct_clinical_arch.sort_index()

In [ ]:
map_overall_oct_clinical['SSL-BYOL'] - map_overall_oct_clinical['ImageNet']

## Lab vs clinical

### SSL

In [ ]:
map_ssl_oct_clinical2 = []
for a in map_ssl_oct_clinical['arch']:
    df_a = map_ssl_oct_clinical[map_ssl_oct_clinical['arch'] == a].drop(columns=['arch', 'train_ratio']).T.reset_index().rename(columns={0: 'clinical'})
    df_a['arch'] = a
    df_a = df_a.set_index(['arch', 'index'])
    map_ssl_oct_clinical2.append(df_a)
map_ssl_oct_clinical2 = pd.concat(map_ssl_oct_clinical2, axis=0)
map_ssl_oct_comp = pd.concat([map_ssl_oct, map_ssl_oct_clinical2], axis=1)
map_ssl_oct_comp = map_ssl_oct_comp.reindex(list(zip(4*['resnet18']+4*['mobilenetv3']+4*['pvtv2b0'], 3*['scratch', 'ImageNet', 'SSL-BYOL', 'SSL-SimCLR'])))
map_ssl_oct_comp  


In [ ]:
(map_ssl_oct_comp*100).round(2)# .to_csv('mAP_ssl.csv')

In [ ]:
map_ssl_oct_comp[map_ssl_oct_comp['clinical'] > map_ssl_oct_comp[15]]

### DINO

In [ ]:
map_dino_oct_clinical2 = []
for a in map_dino_oct_clinical['arch']:
    df_a = map_dino_oct_clinical[map_dino_oct_clinical['arch'] == a].drop(columns=['arch', 'train_ratio']).T.reset_index().rename(columns={0: 'clinical'})
    df_a['arch'] = a
    df_a = df_a.set_index(['arch', 'index'])
    map_dino_oct_clinical2.append(df_a)
map_dino_oct_clinical2 = pd.concat(map_dino_oct_clinical2, axis=0)
map_dino_oct_comp = pd.concat([map_dino_oct, map_dino_oct_clinical2], axis=1)
map_dino_oct_comp = map_dino_oct_comp.reindex(list(zip(2*['vits']+2*['vitb']+2*['vitl'], 3*['DINOv3', 'DINOv3 with LoRA'])))
map_dino_oct_comp


In [ ]:
(map_dino_oct_comp*100).round(2)# .to_csv('mAP_dino.csv')

In [ ]:
map_dino_oct_comp[map_dino_oct_comp['clinical'] > map_dino_oct_comp[15]]

## Classificatifion report and CM

### SSL, lab dataset

In [ ]:
# Get mAP per col
dataset = 'oct'
report_per_arch = {}
for arch in model_archs.keys():
    # Load all pred files
    # Define base folder paths
    supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{arch}')
    byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{arch}')
    simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{arch}')

    # Find pred files
    supervised_pred_files = find_pred_files(supervised_folder, dataset)
    byol_pred_files = find_pred_files(byol_folder, dataset)
    simclr_pred_files = find_pred_files(simclr_folder, dataset)

    # Split random vs imagenet preds
    supervised_pred_random_files = [p for p in supervised_pred_files if 'random' in p.__str__()]
    supervised_pred_imagenet_files = [p for p in supervised_pred_files if 'default' in p.__str__()]

    # Load preds as csv
    supervised_preds_random = load_preds(supervised_pred_random_files, dataset)
    supervised_preds_imagenet = load_preds(supervised_pred_imagenet_files, dataset)
    byol_preds = load_preds(byol_pred_files, dataset)
    simclr_preds = load_preds(simclr_pred_files, dataset)

    all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}

    report_overview = {}
    for w, preds in all_preds.items():
        report_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
        for r in report_overview[w].keys():
            cr_name = f"classification_report/{dataset}_{arch}_{w}_{r}.csv"
            cr_df = pd.DataFrame.from_dict(report_overview[w][r]).T
            cr_df = cr_df.drop(columns=['support'])
            cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            # cr_df.columns = pd.MultiIndex.from_tuples(list(zip(3*[arch], cr_df.columns)))
            cr_df.to_csv(cr_name)
            # print(f"{arch}, {w}, {r}")
            # print(report_overview[w][r])
    
    report_per_arch[arch] = (report_overview)


In [ ]:
# Merge results per arch, for each train_ratio and starting weight
archs = list(report_per_arch.keys())
start_ws = [list(d.keys()) for d in report_per_arch.values()][0] # Starting weights
trs = [list(d1.keys()) for d in report_per_arch.values() for d1 in d.values()][0] # train_ratios

for w in start_ws:
    for tr in trs:
        cr_name = f"classification_report_merged_arch/{dataset}_all_arch_{w}_{tr}.csv"
        cr_arch_all = []
        for arch in archs:
            cr_df = pd.DataFrame.from_dict(report_per_arch[arch][w][tr]).T
            cr_df = cr_df.drop(columns=['support'])
            # cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.columns = pd.MultiIndex.from_tuples(list(zip(3*[model_archs[arch]], cr_df.columns)))
            cr_arch_all.append(cr_df)
        # Merge and format
        cr_arch_all = pd.concat(cr_arch_all, axis=1)
        cr_arch_all_og = cr_arch_all.copy()
        cr_arch_all = cr_arch_all.swaplevel(0, 1, axis=1)
        cr_arch_all = cr_arch_all[pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"]).tolist()]
        cr_arch_all.columns = cr_arch_all.columns.set_names(["Metric", "Encoder"])
        # cr_arch_all.columns = pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"])
        cr_arch_all.to_csv(cr_name)



### SSL clinical dataset

In [ ]:
# Get mAP per col
dataset = 'oct_clinical'
report_per_arch = {}
for arch in model_archs.keys():
    # Load all pred files
    # Define base folder paths
    supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{arch}')
    byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{arch}')
    simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{arch}')

    # Find pred files
    supervised_pred_files = find_pred_files(supervised_folder, dataset)
    byol_pred_files = find_pred_files(byol_folder, dataset)
    simclr_pred_files = find_pred_files(simclr_folder, dataset)

    # Split random vs imagenet preds
    supervised_pred_random_files = [p for p in supervised_pred_files if 'random' in p.__str__()]
    supervised_pred_imagenet_files = [p for p in supervised_pred_files if 'default' in p.__str__()]

    # Load preds as csv
    supervised_preds_random = load_preds(supervised_pred_random_files, dataset)
    supervised_preds_imagenet = load_preds(supervised_pred_imagenet_files, dataset)
    byol_preds = load_preds(byol_pred_files, dataset)
    simclr_preds = load_preds(simclr_pred_files, dataset)

    # Get mAP per col
    all_preds = {'scratch': supervised_preds_random, 'ImageNet': supervised_preds_imagenet, 'SSL-BYOL': byol_preds, 'SSL-SimCLR': simclr_preds}
    report_overview = {}
    for w, preds in all_preds.items():
        report_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
        for r in report_overview[w].keys():
            cr_name = f"classification_report/{dataset}_{arch}_{w}_{r}.csv"
            cr_df = pd.DataFrame.from_dict(report_overview[w][r]).T
            cr_df = cr_df.drop(columns=['support'])
            cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.to_csv(cr_name)
            # print(f"{arch}, {w}, {r}")
            # print(report_overview[w][r])
    
    report_per_arch[arch] = (report_overview)
    


In [ ]:
# Merge results per arch, for each train_ratio and starting weight
archs = list(report_per_arch.keys())
start_ws = [list(d.keys()) for d in report_per_arch.values()][0] # Starting weights
trs = [list(d1.keys()) for d in report_per_arch.values() for d1 in d.values()][0] # train_ratios

for w in start_ws:
    for tr in trs:
        cr_name = f"classification_report_merged_arch/{dataset}_all_arch_{w}_{tr}.csv"
        cr_arch_all = []
        for arch in archs:
            cr_df = pd.DataFrame.from_dict(report_per_arch[arch][w][tr]).T
            cr_df = cr_df.drop(columns=['support'])
            # cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.columns = pd.MultiIndex.from_tuples(list(zip(3*[model_archs[arch]], cr_df.columns)))
            cr_arch_all.append(cr_df)
        # Merge and format
        cr_arch_all = pd.concat(cr_arch_all, axis=1)
        cr_arch_all_og = cr_arch_all.copy()
        cr_arch_all = cr_arch_all.swaplevel(0, 1, axis=1)
        cr_arch_all = cr_arch_all[pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"]).tolist()]
        cr_arch_all.columns = cr_arch_all.columns.set_names(["Metric", "Encoder"])
        # cr_arch_all.columns = pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"])
        cr_arch_all.to_csv(cr_name)




### DINO lab dataset

In [ ]:
# Get mAP per col
dataset = 'oct'
report_per_arch = {}
for arch in dino_archs.keys():
    # Load all pred files
    # Define base folder paths
    dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{arch}')

    # Find pred files
    dino_pred_files = find_pred_files(dino_folder, dataset)
    # Split LoRA and noLoRA preds
    dino_pred_lora_files = [p for p in dino_pred_files if not 'noLoRA' in p.__str__()]
    dino_pred_noLora_files = [p for p in dino_pred_files if 'noLoRA' in p.__str__()]

    # Load preds as csv
    dino_preds_lora = load_preds(dino_pred_lora_files, dataset)
    dino_preds_noLora = load_preds(dino_pred_noLora_files, dataset)

    # Get mAP per col
    all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}

    report_overview = {}
    for w, preds in all_preds.items():
        report_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
        for r in report_overview[w].keys():
            cr_name = f"classification_report/{dataset}_{arch}_{w}_{r}.csv"
            cr_df = pd.DataFrame.from_dict(report_overview[w][r]).T
            cr_df = cr_df.drop(columns=['support'])
            cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.to_csv(cr_name)
            # print(f"{arch}, {w}, {r}")
            # print(report_overview[w][r])
    
    report_per_arch[arch] = (report_overview)
    
    
    

In [ ]:
# Merge results per arch, for each train_ratio and starting weight
archs = list(report_per_arch.keys())
start_ws = [list(d.keys()) for d in report_per_arch.values()][0] # Starting weights
trs = [list(d1.keys()) for d in report_per_arch.values() for d1 in d.values()][0] # train_ratios

for w in start_ws:
    for tr in trs:
        cr_name = f"classification_report_merged_arch/{dataset}_all_arch_{w}_{tr}.csv"
        cr_arch_all = []
        for arch in archs:
            cr_df = pd.DataFrame.from_dict(report_per_arch[arch][w][tr]).T
            cr_df = cr_df.drop(columns=['support'])
            # cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.columns = pd.MultiIndex.from_tuples(list(zip(3*[dino_archs[arch]], cr_df.columns)))
            cr_arch_all.append(cr_df)
        # Merge and format
        cr_arch_all = pd.concat(cr_arch_all, axis=1)
        cr_arch_all_og = cr_arch_all.copy()
        cr_arch_all = cr_arch_all.swaplevel(0, 1, axis=1)
        cr_arch_all = cr_arch_all[pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(dino_archs.values())], names=["Metric", "Encoder"]).tolist()]
        cr_arch_all.columns = cr_arch_all.columns.set_names(["Metric", "Encoder"])
        # cr_arch_all.columns = pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"])
        cr_arch_all.to_csv(cr_name)




### DINO clinical dataset

In [ ]:
# Get mAP per col
dataset = 'oct_clinical'
report_per_arch = {}
for arch in dino_archs.keys():
    # Load all pred files
    # Define base folder paths
    dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{arch}')

    # Find pred files
    dino_pred_files = find_pred_files(dino_folder, dataset)
    # print(f"{arch}: {len(dino_pred_files)}")
    # print(dino_pred_files)
    # Split LoRA and noLoRA preds
    dino_pred_lora_files = [p for p in dino_pred_files if not 'noLoRA' in p.__str__()]
    dino_pred_noLora_files = [p for p in dino_pred_files if 'noLoRA' in p.__str__()]
    # print(f"{arch}: {len(dino_pred_noLora_files)}")
    # Load preds as csv
    dino_preds_lora = load_preds(dino_pred_lora_files, dataset)
    dino_preds_noLora = load_preds(dino_pred_noLora_files, dataset)
    # print(f"{arch}: {len(dino_preds_noLora)}")

    # Get mAP per col
    all_preds = {'DINOv3': dino_preds_noLora, 'DINOv3 with LoRA': dino_preds_lora}
    report_overview = {}
    for w, preds in all_preds.items():
        report_overview[w] = get_f1_score_per_col(preds, keep_weight_init=False)
        for r in report_overview[w].keys():
            cr_name = f"classification_report/{dataset}_{arch}_{w}_{r}.csv"
            cr_df = pd.DataFrame.from_dict(report_overview[w][r]).T
            cr_df = cr_df.drop(columns=['support'])
            cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.to_csv(cr_name)
            # print(f"{arch}, {w}, {r}")
            # print(report_overview[w][r])

    report_per_arch[arch] = (report_overview)

In [ ]:
# Merge results per arch, for each train_ratio and starting weight
archs = list(report_per_arch.keys())
start_ws = [list(d.keys()) for d in report_per_arch.values()][0] # Starting weights
trs = [list(d1.keys()) for d in report_per_arch.values() for d1 in d.values()][0] # train_ratios

for w in start_ws:
    for tr in trs:
        cr_name = f"classification_report_merged_arch/{dataset}_all_arch_{w}_{tr}.csv"
        cr_arch_all = []
        for arch in archs:
            cr_df = pd.DataFrame.from_dict(report_per_arch[arch][w][tr]).T
            cr_df = cr_df.drop(columns=['support'])
            # cr_df.index = cr_df.index.set_names('label')
            cr_df.loc['accuracy', ['precision', 'recall']] = np.nan
            cr_df = (cr_df*100).round(2)
            cr_df.columns = pd.MultiIndex.from_tuples(list(zip(3*[dino_archs[arch]], cr_df.columns)))
            cr_arch_all.append(cr_df)
        # Merge and format
        cr_arch_all = pd.concat(cr_arch_all, axis=1)
        cr_arch_all_og = cr_arch_all.copy()
        cr_arch_all = cr_arch_all.swaplevel(0, 1, axis=1)
        cr_arch_all = cr_arch_all[pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(dino_archs.values())], names=["Metric", "Encoder"]).tolist()]
        cr_arch_all.columns = cr_arch_all.columns.set_names(["Metric", "Encoder"])
        # cr_arch_all.columns = pd.MultiIndex.from_product([['precision', 'recall', 'f1-score'], list(model_archs.values())], names=["Metric", "Encoder"])
        cr_arch_all.to_csv(cr_name)



## CURAC

### Load predictions

In [2]:
def get_lab_splits(map_df_paths):
    # Find all split combinations
    map_dfs = pd.concat([pd.read_csv(pathlib.Path(p)) for p in map_df_paths], axis=0)
    map_dfs['img_relative_path'] = [pathlib.Path(p) for p in map_dfs['img_relative_path']]
    map_dfs.loc[:, 'area'] = [p.parts[-2] for p in map_dfs['img_relative_path']]

    # Get list of all current labels
    lbl_list = map_dfs['label'].sort_values().unique()

    # Find all combinations of 3 within the labels
    splits = list(itertools.combinations(lbl_list, 3))
    return splits

def merge_pred_csv_curac(pred_paths:list, cv_split_str: str)->pd.DataFrame:
    sig = nn.Sigmoid()
    preds = {p: pd.read_csv(p) for p in pred_paths}
    preds_all = pd.DataFrame()
    for p, df in preds.items():
        df['subset'] = 'test'
        if 'split' in p.stem:
            # p = byol_preds[0].stem.replace(cv_split_str, '')
            p = p.parent.joinpath(f"{p.stem.replace(cv_split_str, '')}{p.suffix}")
            # p = p.parent.joinpath(f'{'_'.join(p.stem.split('_')[:-1])}{p.suffix}')
            if 'finetune' in p.stem:
                p = p.parent.joinpath(f"{p.stem.replace('_finetune', '')}{p.suffix}")
        if 'noLoRA' in p.stem:
            p = p.parent.joinpath(f'{'_'.join(p.stem.split('_')[:-1])}{p.suffix}')
        if ('random' in p.stem) or ('default' in p.stem):
            starting_weights = p.stem.split('_')[-1]
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-2]))
            if 'finetune' in p.stem:
                starting_weights = p.stem.split('_')[-1]
                ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-2]))
        else:
            starting_weights = p.parent.parent.stem
            ratio_sup = int(re.sub(r'[^\d]+', '', p.stem.split('_')[-1]))
        col_name = f"{starting_weights}_{re.sub(r'[^\w\s]', '', f'{ratio_sup:03d}')}"
        # print(col_name)
        new_col_names = {c: f'{c}_{col_name}' for c in df.columns if 'prob' in c}
        new_col_names.update({'pred': f'pred_{col_name}'})
        df = df.rename(columns=new_col_names)
        if len([c for c in df.columns if 'prob' in c]) == 1:
            prob_col = [c for c in df.columns if 'prob' in c][0]
            df[prob_col] = [sig(torch.tensor(prob)) for prob in df[prob_col]]
            df[prob_col] = [float(prob) for prob in df[prob_col]]
        # df = df.set_index([c for c in df.columns if 'pred|prob' not in c])
        df = df.set_index([c for c in df.columns if not re.search('pred|prob', c)])
        if len(preds_all) == 0:
            preds_all = df.copy()
        else:
            preds_all = pd.merge(preds_all, df, left_index=True, right_index=True, how='outer')
    preds_all = preds_all.reindex(sorted(preds_all.columns), axis=1)
    return preds_all

def get_map_per_col(df: pd.DataFrame, keep_weight_init=True) -> dict:
    prob_cols = [c for c in df.columns if 'prob' in c]
    ratios = list(set([pc.split('_')[-1] for pc in prob_cols]))
    ratios.sort()
    df = df.reset_index()
    map_dict = {}
    for r in ratios:
        r_cols = [c for c in prob_cols if r in c]
        # print(r_cols)
        df_r = df.dropna(subset=r_cols)
        map_r = average_precision_score(df_r['label'].to_numpy(), df_r[r_cols].to_numpy(), average='macro') # Mean of F1 scores per class
        if keep_weight_init:
            new_r = '_'.join([s for i, s in enumerate(r_cols[0].split('_')) if i != 1])
            map_dict[new_r] = map_r
        else:
            new_r = r.split('_')[-1]
            map_dict[new_r] = map_r
    return map_dict

def get_metric_per_col(df: pd.DataFrame, metric: str, keep_weight_init=True) -> dict:
    pred_cols = [c for c in df.columns if 'pred' in c]
    df = df.reset_index()
    report_dict = {}
    for c in pred_cols:
        # df_c = df.dropna(subset=[c], axis=0)
        df_c = df.copy()
        # f1s_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average=None) # F1 score per class
        # f1_c = f1_score(df_c['label'].tolist(), df_c[c].tolist(), average='macro') # Mean of F1 scores per class
        # report_c = classification_report(np.array(df_c['label']), np.array(df_c[c]), 
        #                                  target_names=df_c['label_str'].unique().tolist(), 
        #                                  output_dict=True,
        #                                  digits=4, zero_division=np.nan)
        if metric == 'f1-score':
            # report_c = report_c['macro avg']['f1-score']
            report_c = f1_score(np.array(df_c['label']), np.array(df_c[c]), average='macro')
        elif metric == 'accuracy':
            # report_c = report_c['accuracy']
            report_c = accuracy_score(np.array(df_c['label']), np.array(df_c[c]))
        if keep_weight_init:
            report_dict[c] = report_c
        else:
            new_c = c.split('_')[-1]
            report_dict[new_c] = report_c
            # print(f"{c}, {new_c} {metric}: {report_c}")
            # print(f"{len(df_c[df_c['label'] == df_c[c]])}/{len(df_c)} acc ({len(df_c[df_c['label'] == df_c[c]])/len(df_c)})")
    return report_dict


In [3]:
# map_df_paths = list(pathlib.Path(r"C:\Users\Ana-Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20").glob('*.csv'))
map_df_paths = list(pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20").glob('*.csv'))
# map_df_paths = list(pathlib.Path(r"/data/Boudreault/OCT_lab_data/512mscans_noNoise_sample20").glob('*.csv'))
map_df_paths = [p for p in map_df_paths if '_Classes3_' in p.__str__()]

model_archs = {'resnet18': 'ResNet-18', 'pvtv2b0': 'PVTv2-b0', 'mobilenetv3': 'MobileNetV3'}
# dino_archs = {'vits': 'ViT-S', 'vitb': 'ViT-B', 'vitl': 'ViT-L'}
model_arch = list(model_archs.keys())[0]
# dino_arch = list(dino_archs.keys())[2]
datasets = {'oct': 'lab', 'oct_clinical': 'clinical'}
dataset = list(datasets.keys())[0]
metrics = ['mAP', 'f1-score', 'accuracy']
ylabel = {m: m_str for m, m_str in zip(metrics, ['Macro mean average precision', 'Macro F1 score', 'Accuracy'])}
metric = metrics[0]

# Get splits
lbl_abbs = {'chicken_heart_muscle': 'chm',
                    'lamb_heart_fat': 'lhf',
                    'lamb_heart_muscle': 'lhm',
                    'lamb_liver': 'll',
                    'lamb_testicle': 'lt'}
splits = get_lab_splits(map_df_paths)


# Define base folder paths
supervised_folder = pathlib.Path.cwd().joinpath('supervised').joinpath(f'weights_{model_arch}')
supervised_pretrain_folder = pathlib.Path.cwd().joinpath('pretrain_supervised').joinpath(f'weights_{model_arch}')
byol_folder = pathlib.Path.cwd().joinpath('BYOL').joinpath(f'weights_{model_arch}')
simclr_folder = pathlib.Path.cwd().joinpath('SimCLR').joinpath(f'weights_{model_arch}')
# dino_folder = pathlib.Path.cwd().joinpath('DINOv3').joinpath(f'weights_{dino_arch}')


In [ ]:
map_dfs = pd.concat([pd.read_csv(pathlib.Path(p)) for p in map_df_paths], axis=0)
map_dfs['img_relative_path'] = [pathlib.Path(p) for p in map_dfs['img_relative_path']]
map_dfs.loc[:, 'area'] = [p.parts[-2] for p in map_dfs['img_relative_path']]
map_dfs['area'].sort_values().unique().tolist()

### Metric plots

In [4]:
# For each split, load supervised, SimCLR and BYOL - one plot
metric = metrics[0]
fig, axs = plt.subplots(nrows=2, ncols=5, figsize=(14, 7), sharey=True)
fig.subplots_adjust(left=0.05, bottom=0.076, right=0.988, top=0.874, wspace=0.262, hspace=0.44)
matplotlib.rcParams['font.size'] = 12

metric_per_split = {}
print(f"Calculating {metric} per split")
for si, (cv_split, ax) in enumerate(zip(splits, axs.ravel())):
    cv_split_str = f"_split_{'_'.join([lbl_abbs[l] for l in cv_split])}"
    preds_random = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'random' in p.__str__()], cv_split_str)
    preds_imagenet = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'default' in p.__str__()], cv_split_str)
    preds_pretrain = merge_pred_csv_curac([p for p in supervised_pretrain_folder.glob(f"*{cv_split_str}*.csv") if 'finetune' in p.__str__()], cv_split_str)
    preds_byol = merge_pred_csv_curac(list(byol_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)
    preds_simclr = merge_pred_csv_curac(list(simclr_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)
    
    all_preds = {'scratch': preds_random, 'ImageNet': preds_imagenet, 'SSL-SimCLR': preds_simclr, 'SSL-BYOL': preds_byol, 'OCT-Pretrain': preds_pretrain}
    # all_preds = {'scratch': preds_random, 'ImageNet': preds_imagenet, 'SSL-SimCLR': preds_simclr, 'SSL-BYOL': preds_byol}
    # markers = {w: m for w, m in zip(all_preds.keys(), ['v', '^', '<', '>'])}
    markers = ['v', '^', '<', '>', '1']
    # markers = ['v', '^', '<', '>']
    # Get mAP per col
    if metric == 'mAP':
        map_overview = {}
        for w, preds in all_preds.items():
            map_overview[w] = get_map_per_col(preds, keep_weight_init=False)
        map_overview = pd.DataFrame.from_dict(map_overview, orient='columns')
        map_overview.index = [int(i) for i in map_overview.index]
        map_overview = map_overview.reset_index().rename(columns={'index': 'train_ratio'})
        metric_per_split[f"{si}: {', '.join(cv_split)}"] = map_overview.set_index('train_ratio').to_dict()
        # print(map_overview)
        # plt_title = f"mAP on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder\nLabels: {[l for l in lbl_abbs.keys() if l not in cv_split]}"
        # plt_title = f"{[l for l in lbl_abbs.keys() if l not in cv_split]}"
        plt_title = '\n'.join([l for l in lbl_abbs.keys() if l not in cv_split])
        map_overview.plot.line(x='train_ratio', y=[c for c in map_overview if c != 'train_ratio'], 
                            linestyle=":",
                            title=plt_title,
                            xlabel=r'% of the training set',
                            ylabel='Macro mean average precision',
                            # marker='o',
                            ax=ax)
        for i, line in enumerate(ax.get_lines()):
            line.set_marker(markers[i])
        ax.legend()
        # plt.savefig(f'map_{model_arch}_{dataset}.png', dpi=500)
        # plt.show()

    # Get F1-score 
    if metric == 'f1-score':
        f1_overview = {}
        for w, preds in all_preds.items():
            f1_overview[w] = get_metric_per_col(preds, 'f1-score', keep_weight_init=False)
        f1_overview = pd.DataFrame.from_dict(f1_overview, orient='columns')
        f1_overview.index = [int(i) for i in f1_overview.index]
        f1_overview = f1_overview.reset_index().rename(columns={'index': 'train_ratio'})
        metric_per_split[f"{si}: {', '.join(cv_split)}"] = f1_overview.set_index('train_ratio').to_dict()
        # print(f1_overview)
        # plt_title = f"F1 score on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder\nLabels: {[l for l in lbl_abbs.keys() if l not in cv_split]}"
        # plt_title = f"{[l for l in lbl_abbs.keys() if l not in cv_split]}"
        plt_title = '\n'.join([l for l in lbl_abbs.keys() if l not in cv_split])
        f1_overview.plot.line(x='train_ratio', y=[c for c in f1_overview if c != 'train_ratio'], 
                            linestyle=":",
                            title=plt_title,
                            xlabel=r'% of the training set',
                            ylabel='Macro F1 score',
                            # marker='o',
                            ax=ax)
        for i, line in enumerate(ax.get_lines()):
            line.set_marker(markers[i])
        ax.legend()
        # plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
        # plt.show()

    # Get Accuracy
    if metric == 'accuracy':
        acc_overview = {}
        for w, preds in all_preds.items():
            acc_overview[w] = get_metric_per_col(preds, 'accuracy', keep_weight_init=False)
        acc_overview = pd.DataFrame.from_dict(acc_overview, orient='columns')
        acc_overview.index = [int(i) for i in acc_overview.index]
        acc_overview = acc_overview.reset_index().rename(columns={'index': 'train_ratio'})
        metric_per_split[f"{si}: {', '.join(cv_split)}"] = acc_overview.set_index('train_ratio').to_dict()
        # print(acc_overview)
        # plt_title = f"F1 score on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder\nLabels: {[l for l in lbl_abbs.keys() if l not in cv_split]}"
        # plt_title = f"Labels: {[l for l in lbl_abbs.keys() if l not in cv_split]}"
        plt_title = '\n'.join([l for l in lbl_abbs.keys() if l not in cv_split])
        # plt_title = f"{[l for l in lbl_abbs.keys() if l not in cv_split]}"
        acc_overview.plot.line(x='train_ratio', y=[c for c in acc_overview if c != 'train_ratio'], 
                            linestyle=":",
                            title=plt_title,
                            xlabel=r'% of the training set',
                            ylabel='Accuracy',
                            # marker='o',
                            ax=ax)
        for i, line in enumerate(ax.get_lines()):
            line.set_marker(markers[i])
        ax.legend()
        # ax.set_title('v = 1',fontweight="bold", size=20)
        # ax.set_title(plt_title, size=10)
        # ax.set_xlabel(r'% of the training set', fontsize = 10)
        # ax.set_ylabel('Macro F1 score', fontsize = 10.0)
    
    # Format figure
    if si > 0:
        ax.get_legend().remove()
    # plt.savefig(f'F1score_{model_arch}_{dataset}.png', dpi=500)
# plt_suptitle = f"{metric} on the {datasets[dataset]} dataset using a {model_archs[model_arch]} encoder"
plt_suptitle = f"Classification {metric} after fine-tuning starting from different initial weights"
plt.suptitle(plt_suptitle, fontsize=14)
# plt.tight_layout()
# plt.savefig(f'results_{metric}_{model_arch}_{dataset}_V2.png', dpi=500)
plt.show()
# Disp table
metric_per_split = pd.DataFrame.from_dict({(i,j): metric_per_split[i][j] 
                           for i in metric_per_split.keys() 
                           for j in metric_per_split[i].keys()},
                       orient='index')
metric_per_split = metric_per_split.rename_axis(index=["ssl labels", "starting weights"])
new_weight_str = {w: f"{i}:{w}" for i, w in enumerate(metric_per_split.index.get_level_values('starting weights').unique().tolist())}
metric_per_split = metric_per_split.reset_index()
metric_per_split['starting weights'] = [new_weight_str[w] for w in metric_per_split['starting weights']]
metric_per_split = metric_per_split.sort_values(['starting weights', 'ssl labels'])
metric_per_split = metric_per_split.set_index(['ssl labels', 'starting weights'])
# metric_per_split


Calculating mAP per split


In [ ]:
df = metric_per_split.reset_index().copy()
df.groupby('starting weights').agg({c: ['mean', 'std'] for c in df.columns if isinstance(c, int)})

In [ ]:
metric_per_split[metric_per_split.index.get_level_values('starting weights').str.contains('SSL')].sort_values(5)
# metric_per_split[~metric_per_split.index.get_level_values('starting weights').str.contains('SSL')].sort_values(5, ascending=False)

In [ ]:
# Plot 5% per split
matplotlib.rcParams['font.size'] = 12
metric_per_split5 = pd.pivot_table(metric_per_split[[5]].reset_index(), index='starting weights', columns='ssl labels', values=5)
metric_per_split5.index = [re.split(':', idx)[1] for idx in metric_per_split5.index]
# metric_per_split5.columns
bar_colors = ['blue', 'orange', 'green', 'red']

fig, axs = plt.subplots(nrows=2, ncols=5, figsize=(14, 7), sharey=True)
fig.subplots_adjust(left=0.05, bottom=0.076, right=0.988, top=0.874, wspace=0.262, hspace=0.44)

for si, (cv_split, ax) in enumerate(zip(splits, axs.ravel())):
    plt_title = '\n'.join([l for l in lbl_abbs.keys() if l not in cv_split])
    col = f"{si}: {', '.join([l for l in lbl_abbs.keys() if l in cv_split])}"
    metric_per_split5[[col]].plot.bar(ax=ax, 
                                      title=plt_title, 
                                      color=['blue', 'orange', 'green', 'red'],
                                      )
    ax.tick_params(labelrotation=45)# , ha='right')
    ax.get_legend().remove()
plt_suptitle = f"Classification {metric} after fine-tuning starting from different initial weights"
plt.suptitle(plt_suptitle, fontsize=14)
plt.tight_layout()
# plt.savefig(f'results_{metric}_{model_arch}_{dataset}_5p.png', dpi=500)
plt.show()

In [ ]:
# Plot mAP averaged over all splits using 5% of the data
matplotlib.rcParams['font.size'] = 16
plt_suptitle = f"Classification {metric} after fine-tuning starting from different\ninitial weight averaged over all splits"
metric_per_split5.mean(axis=1).plot.bar(
                                        # title=plt_suptitle, 
                                        ylabel=ylabel[metric],
                                        zorder=3,
                                        figsize=(7.5, 5),
                                        )
plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)
plt.xticks(rotation=0, ha='center')
plt.tight_layout()
plt.savefig(f'results_{metric}_{model_arch}_{dataset}_5p_avg.png', dpi=500)
plt.show()

In [ ]:
# Plot mAP averaged over all splits
matplotlib.rcParams['font.size'] = 16
plt_suptitle = f"Classification {metric} after fine-tuning starting from different\ninitial weight averaged over all splits"
map_overview_avg = metric_per_split.reset_index().drop(columns=['ssl labels']).groupby('starting weights').mean()
map_overview_avg.index = [re.split(':', idx)[1] for idx in map_overview_avg.index]
map_overview_avg = map_overview_avg.T

# map_overview_avg = map_overview_avg.T[[5, 10, 15]].T
ax = map_overview_avg.plot.line(
                            # title=plt_suptitle,
                            # linestyle=":",
                            linestyle='none',
                            xlabel=r'% of the training set',
                            ylabel=ylabel[metric],
                            marker='o',
                            markersize=10,
                            # figsize=(7.5, 5),
                            figsize=(5, 5),
                            ylim=[0.4, 1],
                            zorder=3,
                            )
for i, line in enumerate(ax.get_lines()):
    line.set_marker(markers[i])
plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)
plt.legend(loc='lower right')
plt.tight_layout()
# plt.savefig(f'results_{metric}_{model_arch}_{dataset}_avg.png', dpi=500)
plt.show()

In [19]:
# Box plot with results
matplotlib.rcParams['font.size'] = 16

# Plot options
plt_with_title = False
plt_with_oct_pretrain = False

# ---- Load data ----
df = metric_per_split.reset_index().copy()

# pct_cols = ["5", "10", "15", "20"]  # the % of training set columns
pct_cols = [5, 10, 15, 20]  # the % of training set columns
label_col = "starting weights"
ssl_col = "ssl labels"

# Order + colors matching your original plot (used for the boxes)
if plt_with_oct_pretrain and '4:OCT-Pretrain' in df['starting weights'].unique():
    group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL", "4:OCT-Pretrain"]
    group_display = {
        "0:scratch": "scratch",
        "1:ImageNet": "ImageNet",
        "2:SSL-SimCLR": "SSL-SimCLR",
        "3:SSL-BYOL": "SSL-BYOL",
        "4:OCT-Pretrain": "OCT-Pretrain",
    }
    group_colors = {
        "0:scratch": "#1f77b4",
        "1:ImageNet": "#ff7f0e",
        "2:SSL-SimCLR": "#2ca02c",
        "3:SSL-BYOL": "#d62728",
        "4:OCT-Pretrain": "#f0e68c",
    }

    x_positions = [int(c) for c in pct_cols]
    n_groups = len(group_order)
    box_width = 0.8
    group_gap = 0.24
    cluster_width = n_groups * box_width + (n_groups - 1) * group_gap
else:
    group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL"]
    group_display = {
        "0:scratch": "scratch",
        "1:ImageNet": "ImageNet",
        "2:SSL-SimCLR": "SSL-SimCLR",
        "3:SSL-BYOL": "SSL-BYOL",
    }
    group_colors = {
        "0:scratch": "#1f77b4",
        "1:ImageNet": "#ff7f0e",
        "2:SSL-SimCLR": "#2ca02c",
        "3:SSL-BYOL": "#d62728",
    }

    x_positions = [int(c) for c in pct_cols]
    n_groups = len(group_order)
    box_width = 1.0
    group_gap = 0.3
    cluster_width = n_groups * box_width + (n_groups - 1) * group_gap

rng = np.random.default_rng(0)     # for small horizontal jitter of points
jitter_scale = box_width * 0.15

if plt_with_title:
    fig, ax = plt.subplots(figsize=(9, 6)) # with title
else:
    # fig, ax = plt.subplots(figsize=(7, 5.5)) # no title
    # fig, ax = plt.subplots(figsize=(7.5, 5.5)) # no title
    fig, ax = plt.subplots(figsize=(9, 6)) # no title


for i, group in enumerate(group_order):
    offset = (i - (n_groups - 1) / 2) * (box_width + group_gap)
    sub = df[df[label_col] == group]

    data = [sub[col].values for col in pct_cols]
    positions = [x + offset for x in x_positions]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        # notch=True,
        showfliers=True,
        medianprops=dict(color="black"),
        zorder=2,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(group_colors[group])
        # patch.set_alpha(0.55)          # lighter so the overlaid points stand out


# --- legend for groups (box colors) ---
group_handles = [
    ax.plot([], [], color=group_colors[g], marker="s", linestyle="none",
             markersize=8, label=group_display[g])[0]
    for g in group_order
]
legend_l = ax.legend(handles=group_handles, title="Starting weights",
                    # fontsize=13, bbox_to_anchor=(1, 0.125), # no title
                    # fontsize=14, bbox_to_anchor=(1, 0), # with title
                    loc="lower right", frameon=True)
ax.add_artist(legend_l)

if plt_with_title:
    plt_title = f"Classification {metric} after fine-tuning starting from\ndifferent initial weights over all splits"
    ax.set_title(plt_title)
ax.set_xticks(x_positions)
ax.set_xticklabels([str(x) for x in x_positions])
ax.set_xlim(min(x_positions) - cluster_width / 2 - 1,
            max(x_positions) + cluster_width / 2 + 1)

ax.set_xlabel("% of the finetuning training set")
ax.set_ylabel("Macro mean average precision")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
title_suffix = "_title" if plt_with_title else ""
pretrain_suffix = "_withOctPretrain" if plt_with_oct_pretrain else ""
plt.savefig(f'results_{metric}_{model_arch}_{dataset}_bxplt{title_suffix}{pretrain_suffix}_ppt.png', dpi=500, bbox_inches="tight")
print("saved")
plt.show()

saved


In [ ]:
# Box plot with identical markers and new legend
matplotlib.rcParams['font.size'] = 16

# ---- Toggle: True to render with the title, False to render without ----
plt_with_title = False
plt_with_oct_pretrain = True

# ---- Load data ----
df = metric_per_split.reset_index().copy()

# pct_cols = ["5", "10", "15", "20"]  # the % of training set columns
pct_cols = [5, 10, 15, 20]  # the % of training set columns
label_col = "starting weights"
ssl_col = "ssl labels"

# Order + colors matching your original plot (used for the boxes)
if plt_with_oct_pretrain and '4:OCT-Pretrain' in df['starting weights'].unique():
    group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL", "4:OCT-Pretrain"]
    group_display = {
        "0:scratch": "scratch",
        "1:ImageNet": "ImageNet",
        "2:SSL-SimCLR": "SSL-SimCLR",
        "3:SSL-BYOL": "SSL-BYOL",
        "4:OCT-Pretrain": "OCT-Pretrain",
    }
    group_colors = {
        "0:scratch": "#1f77b4",
        "1:ImageNet": "#ff7f0e",
        "2:SSL-SimCLR": "#2ca02c",
        "3:SSL-BYOL": "#d62728",
        "4:OCT-Pretrain": "#f0e68c",
    }

    x_positions = [int(c) for c in pct_cols]
    n_groups = len(group_order)
    box_width = 0.8
    group_gap = 0.24
    cluster_width = n_groups * box_width + (n_groups - 1) * group_gap
else:
    group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL"]
    group_display = {
        "0:scratch": "scratch",
        "1:ImageNet": "ImageNet",
        "2:SSL-SimCLR": "SSL-SimCLR",
        "3:SSL-BYOL": "SSL-BYOL",
    }
    group_colors = {
        "0:scratch": "#1f77b4",
        "1:ImageNet": "#ff7f0e",
        "2:SSL-SimCLR": "#2ca02c",
        "3:SSL-BYOL": "#d62728",
    }

    x_positions = [int(c) for c in pct_cols]
    n_groups = len(group_order)
    box_width = 1.0
    group_gap = 0.3
    cluster_width = n_groups * box_width + (n_groups - 1) * group_gap

rng = np.random.default_rng(0)     # for small horizontal jitter of points
jitter_scale = 0 # box_width * 0.15

if plt_with_title:
    fig, ax = plt.subplots(figsize=(9.5, 6.5))  # with title
else:
    # fig, ax = plt.subplots(figsize=(7, 5.5))  # no title
    # fig, ax = plt.subplots(figsize=(7.5, 5.5))  # no title
    fig, ax = plt.subplots(figsize=(9, 6)) # no title

for i, group in enumerate(group_order):
    offset = (i - (n_groups - 1) / 2) * (box_width + group_gap)
    sub = df[df[label_col] == group]

    data = [sub[col].values for col in pct_cols]
    positions = [x + offset for x in x_positions]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black"),
        zorder=2,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(group_colors[group])
        # patch.set_alpha(0.55)          # lighter so the overlaid points stand out

    # overlay individual per-split points, one marker shape per ssl label
    for _, row in sub.iterrows():
        # marker = label_markers[row[ssl_col]]
        # marker_color = label_colors[row[ssl_col]]
        for x, col in zip(positions, pct_cols):
            jitter = rng.uniform(-jitter_scale, jitter_scale)
            ax.scatter(
                x + jitter, row[col],
                marker='.',
                facecolor= 'white', # 'white',
                edgecolor="black",
                linewidths= 0.7,
                s=60,
                zorder=3,
            )

# --- legend for groups (box colors) ---
group_handles = [
    ax.plot([], [], color=group_colors[g], marker="s", linestyle="none",
             markersize=9, label=group_display[g])[0]
    for g in group_order
]
if plt_with_title:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        # fontsize=14, bbox_to_anchor=(1, 0),  # with title
                        loc="lower right", frameon=True)
else:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        # fontsize=13, bbox_to_anchor=(1, 0),  # no title
                        loc="lower right", frameon=True)
ax.add_artist(legend_l)

if plt_with_title:
    plt_title = f"Classification {metric} after fine-tuning starting from\ndifferent initial weights over all splits"
    ax.set_title(plt_title)

ax.set_xticks(x_positions)
ax.set_xticklabels([str(x) for x in x_positions])
ax.set_xlim(min(x_positions) - cluster_width / 2 - 1,
            max(x_positions) + cluster_width / 2 + 1)

ax.set_xlabel("% of the finetuning training set")
ax.set_ylabel("Macro mean average precision")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
title_suffix = "_title" if plt_with_title else ""
pretrain_suffix = "_withOctPretrain" if plt_with_oct_pretrain else ""
plt.savefig(f'results_{metric}_{model_arch}_{dataset}_bxpltMkrSame{title_suffix}{pretrain_suffix}_ppt.png', dpi=500, bbox_inches="tight")
print("saved")
plt.show()


saved


In [ ]:
# Box plot with markers and new legend
matplotlib.rcParams['font.size'] = 16

# ---- Toggle: True to render with the title, False to render without ----
plt_with_title = False

# ---- Load data ----
df = metric_per_split.reset_index().copy()

# pct_cols = ["5", "10", "15", "20"]  # the % of training set columns
pct_cols = [5, 10, 15, 20]  # the % of training set columns
label_col = "starting weights"
ssl_col = "ssl labels"

# Order + colors matching your original plot (used for the boxes)
group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL"]
group_display = {
    "0:scratch": "scratch",
    "1:ImageNet": "ImageNet",
    "2:SSL-SimCLR": "SSL-SimCLR",
    "3:SSL-BYOL": "SSL-BYOL",
}
group_colors = {
    "0:scratch": "#1f77b4",
    "1:ImageNet": "#ff7f0e",
    "2:SSL-SimCLR": '#da70d6', #  "#2ca02c",
    "3:SSL-BYOL": '#f0e68c',  # "#d62728",
}

# One marker per ssl label, kept consistent across all 4 groups so the
# same split is identifiable everywhere it appears.
ssl_labels = sorted(df[ssl_col].unique(), key=lambda s: int(s.split(":")[0]))
marker_shapes = ["o", "s", "^", "v", "D", "P", "X", "*", "h", "<"]
if len(ssl_labels) > len(marker_shapes):
    raise ValueError("More ssl labels than defined marker shapes — add more markers.")
label_markers = dict(zip(ssl_labels, marker_shapes))

# --- rank ssl labels by mean performance on the SSL-initialised rows ---
# (mean over pct_cols first, per row, then averaged across the SSL rows
# for that ssl label — i.e. mean over SSL-SimCLR and SSL-BYOL).
ssl_rows = df[df[label_col].str.contains("SSL")].copy()
ssl_rows["_row_mean"] = ssl_rows[pct_cols].mean(axis=1)
ssl_rank = ssl_rows.groupby(ssl_col)["_row_mean"].mean().sort_values(ascending=False)

n_highlight = 3
best_labels = ssl_rank.index[:n_highlight].tolist()                 # best -> worst
worst_labels = ssl_rank.index[-n_highlight:].tolist()[::-1]         # worst -> best

# darkest -> lightest
green_shades = ["#1b5e20", "#4caf50", "#a5d6a7"]  # best, 2nd best, 3rd best
red_shades = ["#8b0000", "#d62728", "#f4a3a3"]    # worst, 2nd worst, 3rd worst

default_marker_color = "white"
label_colors = {lab: default_marker_color for lab in ssl_labels}
for lab, c in zip(best_labels, green_shades):
    label_colors[lab] = c
for lab, c in zip(worst_labels, red_shades):
    label_colors[lab] = c

# --- build "missing classes" abbreviations for the legend ---
# Each ssl label lists the 3 classes used for that split; there are 5
# classes in total, so 2 are always missing. We abbreviate each missing
# class to the first letter of each of its underscore-separated words
# (e.g. "lamb_heart_fat" -> "lhf", "lamb_liver" -> "ll") and join the two
# abbreviations with "_" (e.g. "ll-lt").
def parse_classes(label):
    # label looks like "0: chicken_heart_muscle, lamb_heart_fat, lamb_heart_muscle"
    return [c.strip() for c in label.split(":", 1)[1].split(",")]

def abbreviate(class_name):
    return "".join(word[0] for word in class_name.split("_"))

all_classes = sorted(set(c for lab in ssl_labels for c in parse_classes(lab)))

label_short = {}
for lab in ssl_labels:
    present = set(parse_classes(lab))
    missing = [c for c in all_classes if c not in present]  # fixed, deterministic order
    # label_short[lab] = f"{lab[0]}: " + "-".join(abbreviate(c) for c in missing)
    label_short[lab] = "-".join(abbreviate(c) for c in missing)

x_positions = [int(c) for c in pct_cols]
n_groups = len(group_order)
box_width = 1.0
group_gap = 0.3
cluster_width = n_groups * box_width + (n_groups - 1) * group_gap

rng = np.random.default_rng(0)     # for small horizontal jitter of points
jitter_scale = box_width * 0.15

if plt_with_title:
    fig, ax = plt.subplots(figsize=(9.5, 6.5))  # with title
else:
    # fig, ax = plt.subplots(figsize=(7, 5.5))  # no title
    fig, ax = plt.subplots(figsize=(6.5, 5.5))  # no title

for i, group in enumerate(group_order):
    offset = (i - (n_groups - 1) / 2) * (box_width + group_gap)
    sub = df[df[label_col] == group]

    data = [sub[col].values for col in pct_cols]
    positions = [x + offset for x in x_positions]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black"),
        zorder=2,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(group_colors[group])
        # patch.set_alpha(0.55)          # lighter so the overlaid points stand out

    # overlay individual per-split points, one marker shape per ssl label
    for _, row in sub.iterrows():
        marker = label_markers[row[ssl_col]]
        marker_color = label_colors[row[ssl_col]]
        for x, col in zip(positions, pct_cols):
            jitter = rng.uniform(-jitter_scale, jitter_scale)
            ax.scatter(
                x + jitter, row[col],
                marker=marker,
                facecolor=marker_color,
                edgecolor="black",
                linewidths=0.7,
                s=35,
                zorder=3,
            )


# --- legend for ssl labels (marker shapes + highlight colors) ---
marker_handles = [
    ax.plot([], [], marker=label_markers[lab], linestyle="none",
             markerfacecolor=label_colors[lab], markeredgecolor="black",
             markersize=7, label=label_short[lab])[0]
    for lab in ssl_labels
]
if plt_with_title:
    legend_m = ax.legend(handles=marker_handles, title="Supervised labels", loc="lower left",
                         ncol=5, frameon=True, fontsize=14, columnspacing=0.4, handletextpad=0.2,  # with title
              )
else:
    legend_m = ax.legend(handles=marker_handles, title="Supervised labels", loc="lower left",
                         ncol=5, frameon=True, fontsize=12, columnspacing=0.4, handletextpad=0.4,  # no title
              )
ax.add_artist(legend_m)

# --- legend for groups (box colors) ---
group_handles = [
    ax.plot([], [], color=group_colors[g], marker="s", linestyle="none",
             markersize=9, label=group_display[g])[0]
    for g in group_order
]
if plt_with_title:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        fontsize=14, bbox_to_anchor=(1, 0),  # with title
                        loc="lower right", frameon=True)
else:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        fontsize=13, bbox_to_anchor=(1, 0.125),  # no title
                        loc="lower right", frameon=True)
ax.add_artist(legend_l)

if plt_with_title:
    plt_title = f"Classification {metric} after fine-tuning starting from\ndifferent initial weights over all splits"
    ax.set_title(plt_title)

ax.set_xticks(x_positions)
ax.set_xticklabels([str(x) for x in x_positions])
ax.set_xlim(min(x_positions) - cluster_width / 2 - 1,
            max(x_positions) + cluster_width / 2 + 1)

ax.set_xlabel("% of the finetuning training set")
ax.set_ylabel("Macro mean average precision")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
title_suffix = "_title" if plt_with_title else ""
plt.savefig(f'results_{metric}_{model_arch}_{dataset}_bxpltMkrAll{title_suffix}.png', dpi=500, bbox_inches="tight")
print("saved")

In [ ]:
# Box plot with best/worst markers and new legend
matplotlib.rcParams['font.size'] = 16

# ---- Toggle: True to render with the title, False to render without ----
plt_with_title = True

# ---- Load data ----
df = metric_per_split.reset_index().copy()

# pct_cols = ["5", "10", "15", "20"]  # the % of training set columns
pct_cols = [5, 10, 15, 20]  # the % of training set columns
label_col = "starting weights"
ssl_col = "ssl labels"

# Order + colors matching your original plot (used for the boxes)
group_order = ["0:scratch", "1:ImageNet", "2:SSL-SimCLR", "3:SSL-BYOL"]
group_display = {
    "0:scratch": "scratch",
    "1:ImageNet": "ImageNet",
    "2:SSL-SimCLR": "SSL-SimCLR",
    "3:SSL-BYOL": "SSL-BYOL",
}
group_colors = {
    "0:scratch": "#1f77b4",
    "1:ImageNet": "#ff7f0e",
    "2:SSL-SimCLR": '#9467bd', #  "#2ca02c",
    "3:SSL-BYOL": '#f0e68c',  # "#d62728",
}

# One marker per ssl label, kept consistent across all 4 groups so the
# same split is identifiable everywhere it appears.
ssl_labels = sorted(df[ssl_col].unique(), key=lambda s: int(s.split(":")[0]))
marker_shapes = ["o", "s", "^", "v", "D", "P", "X", "*", "h", "<"]
if len(ssl_labels) > len(marker_shapes):
    raise ValueError("More ssl labels than defined marker shapes — add more markers.")
label_markers = dict(zip(ssl_labels, marker_shapes))

# --- rank ssl labels by mean performance on the SSL-initialised rows ---
# (mean over pct_cols first, per row, then averaged across the SSL rows
# for that ssl label — i.e. mean over SSL-SimCLR and SSL-BYOL).
ssl_rows = df[df[label_col].str.contains("SSL")].copy()
ssl_rows["_row_mean"] = ssl_rows[pct_cols].mean(axis=1)
ssl_rank = ssl_rows.groupby(ssl_col)["_row_mean"].mean().sort_values(ascending=False)

best_label = ssl_rank.index[0]
worst_label = ssl_rank.index[-1]
highlighted_labels = [best_label, worst_label]  # only these are plotted as points

label_colors = {
    best_label: "#2ca02c",   # green
    worst_label: "#d62728",  # red
}

# best gets an upward triangle, worst a downward triangle
label_markers[best_label] = "^"
label_markers[worst_label] = "v"

# --- build "missing classes" abbreviations for the legend ---
# Each ssl label lists the 3 classes used for that split; there are 5
# classes in total, so 2 are always missing. We abbreviate each missing
# class to the first letter of each of its underscore-separated words
# (e.g. "lamb_heart_fat" -> "lhf", "lamb_liver" -> "ll") and join the two
# abbreviations with "_" (e.g. "ll-lt").
def parse_classes(label):
    # label looks like "0: chicken_heart_muscle, lamb_heart_fat, lamb_heart_muscle"
    return [c.strip() for c in label.split(":", 1)[1].split(",")]

def abbreviate(class_name):
    return "".join(word[0] for word in class_name.split("_"))

def format_class(class_name):
    # "lamb_liver" -> "Lamb liver"
    return class_name.replace("_", " ").capitalize()

all_classes = sorted(set(c for lab in ssl_labels for c in parse_classes(lab)))

label_short = {}
for lab in ssl_labels:
    present = set(parse_classes(lab))
    missing = [c for c in all_classes if c not in present]  # fixed, deterministic order
    label_short[lab] = " - ".join(format_class(c) for c in missing)

x_positions = [int(c) for c in pct_cols]
n_groups = len(group_order)
box_width = 1.0
group_gap = 0.3
cluster_width = n_groups * box_width + (n_groups - 1) * group_gap

rng = np.random.default_rng(0)     # for small horizontal jitter of points
jitter_scale = box_width * 0.15

if plt_with_title:
    fig, ax = plt.subplots(figsize=(9, 6))  # with title
else:
    # fig, ax = plt.subplots(figsize=(7, 5.5))  # no title
    fig, ax = plt.subplots(figsize=(6.5, 5.5))  # no title

for i, group in enumerate(group_order):
    offset = (i - (n_groups - 1) / 2) * (box_width + group_gap)
    sub = df[df[label_col] == group]

    data = [sub[col].values for col in pct_cols]
    positions = [x + offset for x in x_positions]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black"),
        zorder=2,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(group_colors[group])
        # patch.set_alpha(0.55)          # lighter so the overlaid points stand out

    # overlay per-split points only for the best and worst ssl labels
    sub_highlight = sub[sub[ssl_col].isin(highlighted_labels)]
    for mi, (_, row) in enumerate(sub_highlight.iterrows()):
        marker = label_markers[row[ssl_col]]
        marker_color = label_colors[row[ssl_col]]
        for x, col in zip(positions, pct_cols):
            if mi == 0:
                jitter = jitter_scale
            elif mi == 1:
                jitter = -jitter_scale
            else:
                jitter = rng.uniform(-jitter_scale, jitter_scale)
            ax.scatter(
                x + jitter, row[col],
                marker=marker,
                facecolor=marker_color,
                edgecolor="black",
                linewidths=0.7,
                s=50,
                zorder=3,
            )


# --- legend for ssl labels (only best/worst, missing-class names, colored) ---
marker_handles = [
    ax.plot([], [], marker=label_markers[lab], linestyle="none",
             markerfacecolor=label_colors[lab], markeredgecolor="black",
             markersize=7, label=label_short[lab])[0]
    for lab in highlighted_labels
]
if plt_with_title:
    legend_m = ax.legend(handles=marker_handles, title="Best/worst supervised labels", loc="lower left",
                         ncol=1, frameon=True, fontsize=14, columnspacing=0.4, handletextpad=0.2,  # with title
              )
else:
    legend_m = ax.legend(handles=marker_handles, title="Supervised labels", loc="lower left",
                         ncol=1, frameon=True, fontsize=12, columnspacing=0.8, handletextpad=0.4,  # no title
              )
ax.add_artist(legend_m)

# --- legend for groups (box colors) ---
group_handles = [
    ax.plot([], [], color=group_colors[g], marker="s", linestyle="none",
             markersize=8, label=group_display[g])[0]
    for g in group_order
]
if plt_with_title:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        fontsize=14, bbox_to_anchor=(1, 0),  # with title
                        loc="lower right", frameon=True)
else:
    legend_l = ax.legend(handles=group_handles, title="Starting weights",
                        fontsize=13, bbox_to_anchor=(1, 0),  # no title
                        loc="lower right", frameon=True)
ax.add_artist(legend_l)
if plt_with_title:
    plt_title = f"Classification {metric} after fine-tuning starting from\ndifferent initial weights over all splits"
    ax.set_title(plt_title)

ax.set_xticks(x_positions)
ax.set_xticklabels([str(x) for x in x_positions])
ax.set_xlim(min(x_positions) - cluster_width / 2 - 1,
            max(x_positions) + cluster_width / 2 + 1)

ax.set_xlabel("% of the finetuning training set")
ax.set_ylabel("Macro mean average precision")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
title_suffix = "_title" if plt_with_title else ""
plt.savefig(f'results_{metric}_{model_arch}_{dataset}_bxpltMkrBW{title_suffix}.png', dpi=500, bbox_inches="tight")
print("saved")

In [ ]:
df = df[~df['starting weights'].str.contains('4')].copy()

In [ ]:
metric_per_split.reset_index().groupby('starting weights').agg({c: 'mean' for c in metric_per_split.columns}).mean(axis=1)

In [ ]:
metric_per_split

In [ ]:
from scipy.stats import permutation_test

In [ ]:
metric_per_split.reset_index().columns

In [ ]:
from scipy.stats import permutation_test, friedmanchisquare

In [78]:

def analyze_cross_validation(df, target_setting=5):
    """
    Analyzes paired mAP scores across class-combination splits.
    
    Parameters:
    - df: pandas DataFrame with integer keys for percentages (e.g., 5, 10, 15, 20)
    - target_setting: 5 (int) for 5% fine-tuning, or 'average' (str) for across all percentages
    """
    df = df.copy()
    
    # 1. Map columns/metrics based on input types
    percentage_cols = [5, 10, 15, 20]
    if target_setting == 'average':
        df['metric'] = df[percentage_cols].mean(axis=1)
    else:
        df['metric'] = df[int(target_setting)]
        
    # 2. Pivot the table using exact dataset strings
    pivot_df = df.pivot(index='ssl labels', columns='starting weights', values='metric')
    
    # Verify required keys exist in your data framework
    required_cols = ['0:scratch', '1:ImageNet', '2:SSL-SimCLR', '3:SSL-BYOL']
    missing = [c for c in required_cols if c not in pivot_df.columns]
    if missing:
        raise KeyError(f"Could not find these exact labels in your 'starting weights' column: {missing}")

    # 3. Calculate the averaged domain-specific SSL performance per split
    ssl_variants = ['2:SSL-SimCLR', '3:SSL-BYOL']
    pivot_df['ssl_avg'] = pivot_df[ssl_variants].mean(axis=1)
    
    # Isolate vector differences against the ImageNet baseline
    baseline = '1:ImageNet'
    paired_data = pivot_df[['ssl_avg', baseline]].dropna()
    ssl_scores = paired_data['ssl_avg'].values
    baseline_scores = paired_data[baseline].values
    diffs = ssl_scores - baseline_scores
    
    # Compute descriptive variance metrics
    mean_diff = np.mean(diffs)
    fold_variance = np.var(diffs, ddof=1)
    
    # 4. Paired Sign-Flip Permutation Test
    def test_statistic(flipped_diffs):
        return np.mean(flipped_diffs)
    
    perm_res = permutation_test(
        (diffs,), 
        test_statistic, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=9999
    )
    p_val_perm = perm_res.pvalue
    
    # 5. Percentile Bootstrap for 95% Confidence Interval
    rng = np.random.default_rng(42)
    boot_means = [np.mean(rng.choice(diffs, size=len(diffs), replace=True)) for _ in range(10000)]
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    
    # 6. Friedman Test (Inherent Task Difficulty Significance)
    # Checks if ranking distributions across the 10 rows systematically vary by initialization type.
    f_stat, p_val_friedman = friedmanchisquare(
        pivot_df['0:scratch'].values,
        pivot_df['1:ImageNet'].values,
        pivot_df['2:SSL-SimCLR'].values,
        pivot_df['3:SSL-BYOL'].values
    )
    
    # 7. Extract Difficulty Extremes for reporting
    pivot_df['split_difficulty_mean'] = pivot_df[required_cols].mean(axis=1)
    sorted_splits = pivot_df.sort_values(by='split_difficulty_mean')
    
    hardest_split = sorted_splits.index[0]
    hardest_score = sorted_splits['split_difficulty_mean'].iloc[0]
    easiest_split = sorted_splits.index[-1]
    easiest_score = sorted_splits['split_difficulty_mean'].iloc[-1]
    global_split_variance = np.var(sorted_splits['split_difficulty_mean'], ddof=1)

    # --- Print Outputs ---
    print(f"=== METRICS FOR FINETUNING SETTING: {str(target_setting).upper()} ===")
    print(f"Verified rows/splits parsed: {len(pivot_df)}")
    print(f"Mean Delta (SSL Average - ImageNet Baseline): {mean_diff:.4f}")
    print(f"95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")
    print(f"Fold Variance (of difference vector): {fold_variance:.6f}")
    print(f"Paired Permutation Test p-value: {p_val_perm:.4f}")
    print(f"Friedman Test (diff across configurations): Chi2={f_stat:.2f}, p={p_val_friedman:.4f}\n")
    
    print("--- Option A: One-Sentence Summary (Model Comparison) ---")
    sig_text = "significantly outperformed" if p_val_perm < 0.05 else "did not significantly differ from"
    print(f"Across 10 class-combination splits, domain-specific SSL pre-training {sig_text} "
          f"ImageNet initialization (mean \u0394 mAP = {mean_diff:.3f}, 95% CI [{ci_low:.3f}, {ci_high:.3f}], "
          f"fold variance = {fold_variance:.4f}; paired permutation test, p = {p_val_perm:.4f}).\n")
          
    print("--- Option B: One-Sentence Summary (Split/Task Difficulty) ---")
    print(f"Substantial variance in classification task difficulty was observed across the 10 class-combination splits "
          f"(var = {global_split_variance:.4f}; Friedman test \u03c7\u00b2 = {f_stat:.2f}, p = {p_val_friedman:.4f}), "
          f"with average mAP values ranging from {hardest_score:.3f} on the most challenging configuration ({hardest_split}) "
          f"to {easiest_score:.3f} on the least challenging ({easiest_split}).")

# =====================================================================
# EXECUTION COMMANDS:
# =====================================================================
# For the 5% finetuning set setting:
# analyze_cross_validation(df, target_setting=5)

# For the averaged model output across all available percentages:
# analyze_cross_validation(df, target_setting='average')


In [79]:
analyze_cross_validation(df, target_setting='average')

=== METRICS FOR FINETUNING SETTING: AVERAGE ===
Verified rows/splits parsed: 10
Mean Delta (SSL Average - ImageNet Baseline): 0.0797
95% Bootstrap CI: [0.0360, 0.1260]
Fold Variance (of difference vector): 0.005909
Paired Permutation Test p-value: 0.0117
Friedman Test (diff across configurations): Chi2=24.84, p=0.0000

--- Option A: One-Sentence Summary (Model Comparison) ---
Across 10 class-combination splits, domain-specific SSL pre-training significantly outperformed ImageNet initialization (mean Δ mAP = 0.080, 95% CI [0.036, 0.126], fold variance = 0.0059; paired permutation test, p = 0.0117).

--- Option B: One-Sentence Summary (Split/Task Difficulty) ---
Substantial variance in classification task difficulty was observed across the 10 class-combination splits (var = 0.0043; Friedman test χ² = 24.84, p = 0.0000), with average mAP values ranging from 0.774 on the most challenging configuration (9: lamb_heart_muscle, lamb_liver, lamb_testicle) to 0.931 on the least challenging (6: l

In [ ]:
df_ssl = df[df['starting weights'].str.contains('SSL')].copy()
df_ssl['avg'] = df_ssl[[5, 10, 15, 20]].mean(axis=1)
print("Results averaged over all train ratios for SSL pretraining")
df_ssl.groupby('ssl labels').agg({'avg': 'mean'}).sort_values('avg', ascending=True)

In [ ]:
df_sup = df[~df['starting weights'].str.contains('SSL')].copy()
df_sup['avg'] = df_sup[[5, 10, 15, 20]].mean(axis=1)
print("Results averaged over all train ratios for supervised training")
df_sup.groupby('ssl labels').agg({'avg': 'mean'}).sort_values('avg', ascending=True)

In [ ]:
df_all = df.copy()
df_all['avg'] = df_all[[5, 10, 15, 20]].mean(axis=1)
print("Results averaged over all train ratios for all starting weights")
df_all.groupby('ssl labels').agg({'avg': 'mean'}).sort_values('avg', ascending=True)

In [98]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from itertools import product

# df = pd.read_csv("/mnt/user-data/uploads/results_per_split.csv")
# df.rename(columns={"5": 5, "10": 10, "15": 15, "20": 20}, inplace=True)

df["method"] = df["starting weights"].str.split(":").str[1]
splits = [5, 10, 15, 20]

# imagenet = df[df["method"] == "ImageNet"].set_index("ssl labels")[splits]
imagenet = df[df["method"] == "OCT-Pretrain"].set_index("ssl labels")[splits]
simclr   = df[df["method"] == "SSL-SimCLR"].set_index("ssl labels")[splits]
byol     = df[df["method"] == "SSL-BYOL"].set_index("ssl labels")[splits]

shared = imagenet.index.intersection(simclr.index).intersection(byol.index)
imagenet = imagenet.loc[shared]
simclr   = simclr.loc[shared]
byol     = byol.loc[shared]

def paired_permutation_test(a, b, n_permutations=None):
    """
    Exact two-sided paired permutation test.
    For each pair, the sign of the difference can be flipped.
    With n=10, 2^10=1024 permutations are enumerable exactly.
    """
    diffs = a - b
    n = len(diffs)
    observed_stat = np.abs(np.sum(diffs))

    # All 2^n sign flips
    signs = np.array(list(product([-1, 1], repeat=n)))  # shape (2^n, n)
    permuted_stats = np.abs(signs @ diffs)

    p_value = np.mean(permuted_stats >= observed_stat)
    return p_value

rows_wilcoxon = []
rows_permutation = []

for pct in splits:
    in_vals  = imagenet[pct].values
    sim_vals = simclr[pct].values
    byl_vals = byol[pct].values

    # Wilcoxon
    _, p_wil_sim = wilcoxon(in_vals, sim_vals, alternative="two-sided")
    _, p_wil_byl = wilcoxon(in_vals, byl_vals, alternative="two-sided")

    # Paired permutation
    p_per_sim = paired_permutation_test(in_vals, sim_vals)
    p_per_byl = paired_permutation_test(in_vals, byl_vals)

    rows_wilcoxon.append({
        "Fine-tune data (%)": pct,
        "p-value (ImageNet vs SimCLR)": round(p_wil_sim, 4),
        "p-value (ImageNet vs BYOL)":   round(p_wil_byl, 4),
    })
    rows_permutation.append({
        "Fine-tune data (%)": pct,
        "p-value (ImageNet vs SimCLR)": round(p_per_sim, 4),
        "p-value (ImageNet vs BYOL)":   round(p_per_byl, 4),
    })

wil_df = pd.DataFrame(rows_wilcoxon).set_index("Fine-tune data (%)")
per_df = pd.DataFrame(rows_permutation).set_index("Fine-tune data (%)")

print("Wilcoxon Signed-Rank Test")
print("=" * 55)
print(wil_df.to_string())

print("\nPaired Permutation Test (exact, 2^10 = 1024 permutations)")
print("=" * 55)
print(per_df.to_string())

# --- Mean of SSL approaches ---
ssl_mean = (simclr + byol) / 2

rows_wil_mean = []
rows_per_mean = []

for pct in splits:
    in_vals   = imagenet[pct].values
    mean_vals = ssl_mean[pct].values

    _, p_wil = wilcoxon(in_vals, mean_vals, alternative="two-sided")
    p_per    = paired_permutation_test(in_vals, mean_vals)

    rows_wil_mean.append({"Fine-tune data (%)": pct, "p-value (ImageNet vs mean SSL)": round(p_wil, 4)})
    rows_per_mean.append({"Fine-tune data (%)": pct, "p-value (ImageNet vs mean SSL)": round(p_per, 4)})

wil_mean_df = pd.DataFrame(rows_wil_mean).set_index("Fine-tune data (%)")
per_mean_df = pd.DataFrame(rows_per_mean).set_index("Fine-tune data (%)")

print("\nWilcoxon Signed-Rank Test — ImageNet vs mean(SimCLR, BYOL)")
print("=" * 55)
print(wil_mean_df.to_string())

print("\nPaired Permutation Test — ImageNet vs mean(SimCLR, BYOL)")
print("=" * 55)
print(per_mean_df.to_string())

Wilcoxon Signed-Rank Test
                    p-value (ImageNet vs SimCLR)  p-value (ImageNet vs BYOL)
Fine-tune data (%)                                                          
5                                         0.5703                      0.0020
10                                        0.9102                      0.0195
15                                        0.1641                      0.0117
20                                        0.8457                      0.0195

Paired Permutation Test (exact, 2^10 = 1024 permutations)
                    p-value (ImageNet vs SimCLR)  p-value (ImageNet vs BYOL)
Fine-tune data (%)                                                          
5                                         0.9492                      0.0020
10                                        0.6992                      0.0195
15                                        0.1484                      0.0117
20                                        0.8086                    

In [99]:
pd.merge(per_df, per_mean_df, left_index=True, right_index=True)

,p-value (ImageNet vs SimCLR),p-value (ImageNet vs BYOL),p-value (ImageNet vs mean SSL)
Fine-tune data (%),,,
5,0.9492,0.0020,0.1934
10,0.6992,0.0195,0.1328
15,0.1484,0.0117,0.1445
20,0.8086,0.0195,0.3398


### Prediction plots
Plot softmax output for split 3: chicken_heart_muscle, lamb_heart_muscle, lamb_liver at 5% for each starting weight

In [ ]:
metric_per_split[metric_per_split.index.get_level_values('ssl labels').str.contains('3')]

In [ ]:
# Load the predictions per starting weights
cv_split = splits[3]
cv_split_str = f"_split_{'_'.join([lbl_abbs[l] for l in cv_split])}"
preds_random = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'random' in p.__str__()], cv_split_str)
preds_imagenet = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'default' in p.__str__()], cv_split_str)
preds_byol = merge_pred_csv_curac(list(byol_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)
preds_simclr = merge_pred_csv_curac(list(simclr_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)

# cols = [c for c in preds_random.columns if '005' in c]
all_preds = {'scratch': preds_random, 'ImageNet': preds_imagenet, 'SSL-SimCLR': preds_simclr, 'SSL-BYOL': preds_byol}




In [ ]:
# Get histogram per starting weight
matplotlib.rcParams['font.size'] = 12
fig, axs = plt.subplots(nrows=4, ncols=1, figsize=(7, 7), sharex=True)
for ax, (w, p) in zip(axs.ravel(), all_preds.items()):
    # Keep relevant column
    prob_col = [c for c in p.columns if ('005' in c)&('prob' in c)][0] 
    classes_dict = p.reset_index()[['label_str', 'label']].drop_duplicates(subset=['label'], keep='first').set_index('label_str').to_dict()['label']
    df_plt = p[prob_col].reset_index()[['label_str', prob_col]].copy()
    for lbl, lbl_id in classes_dict.items():
        df_lbl = df_plt.loc[df_plt['label_str'] == lbl, prob_col]
        lbl_str = f"{int(lbl_id)}: {re.sub('_', ' ', lbl)}"
        ax.hist(df_lbl, bins=20, alpha=0.5, label=lbl_str)
    ax.set_title(w)
    ax.legend(title='Label')
    if w != 'scratch': # 'SSL-BYOL':
        ax.get_legend().remove()
plt_suptitle = f"Class probability after fine-tuning starting from different\ninitial weights using 5% of the data"
plt.suptitle(plt_suptitle, fontsize=14)
plt.tight_layout()
plt.savefig(f'results_prob_{model_arch}{cv_split_str}.png', dpi=500)
plt.show()
plt.close()

### Image count

In [ ]:
# Load the predictions per starting weights
cv_split = splits[3]
cv_split_str = f"_split_{'_'.join([lbl_abbs[l] for l in cv_split])}"
preds_random = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'random' in p.__str__()], cv_split_str)
preds_imagenet = merge_pred_csv_curac([p for p in supervised_folder.glob(f"*{cv_split_str}.csv") if 'default' in p.__str__()], cv_split_str)
preds_byol = merge_pred_csv_curac(list(byol_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)
preds_simclr = merge_pred_csv_curac(list(simclr_folder.glob(f"*{cv_split_str}.csv")), cv_split_str)

# cols = [c for c in preds_random.columns if '005' in c]
all_preds = {'scratch': preds_random, 'ImageNet': preds_imagenet, 'SSL-SimCLR': preds_simclr, 'SSL-BYOL': preds_byol}




### Trajectory plots

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3d projection)

In [ ]:
"""
Plot a robot trajectory from a delta-position log file and save a rotating
3D view as a GIF — Jupyter-friendly version.

Expected file format (whitespace-separated, no header):
    diffx  diffy  diffz  q1  q2  q3  q4  speed

Each row gives the CHANGE in position for that step (relative to the
previous point), starting from an assumed origin of (0, 0, 0). The actual
trajectory is the cumulative sum of the diffx/diffy/diffz columns.

Just edit FILEPATH / GIF_PATH below and run the cell.
"""

# ---- Configuration ----
FILEPATH = Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\Matlab\ABB_control\abb_trajectories\sine_5x5y_s3_zamp05.txt")  # <- change this to your file
GIF_PATH = Path(f"{FILEPATH.stem}.gif")
COLOR_BY = "index"  # "index" or "speed"
N_FRAMES = 60  # frames per full rotation
FPS = 5
ELEVATION = 20  # fixed viewing elevation while rotating around Z

# ---- Load data ----
data = np.loadtxt(FILEPATH)
if data.ndim == 1:
    data = data.reshape(1, -1)

diffx, diffy, diffz = data[:, 0], data[:, 1], data[:, 2]
speed = data[:, 7] if data.shape[1] > 7 else None

# Actual position = cumulative sum of the per-step deltas, starting at (0,0,0)
x = np.concatenate(([0.0], np.cumsum(diffx)))
y = np.concatenate(([0.0], np.cumsum(diffy)))
z = np.concatenate(([0.0], np.cumsum(diffz)))

# ---- Plot (3D view only) ----
fig = plt.figure(figsize=(7, 5.5))
ax3d = fig.add_subplot(1, 1, 1, projection="3d")

if COLOR_BY == "speed" and speed is not None:
    c = np.concatenate(([speed[0]], speed))
    label = "speed"
else:
    c = np.arange(len(x))
    label = "step index"

sc = ax3d.scatter(x, y, z, c=c, cmap="viridis", s=8)
fig.colorbar(sc, ax=ax3d, shrink=0.6, label=label)
ax3d.plot(x, y, z, color="gray", linewidth=0.6, alpha=0.6)
# ax3d.scatter(x[0], y[0], z[0], color="green", s=60, label="start")
# ax3d.scatter(x[-1], y[-1], z[-1], color="red", s=60, label="end")
ax3d.set_xlabel("X")
ax3d.set_ylabel("Y")
ax3d.set_zlabel("Z")
# ax3d.set_title(FILEPATH.name)
# ax3d.legend()
plt.tight_layout()

# ---- Animate rotation around the Z axis ----
def update(frame):
    azim = 360 * frame / N_FRAMES
    ax3d.view_init(elev=ELEVATION, azim=azim)
    return (sc,)


anim = FuncAnimation(fig, update, frames=N_FRAMES, interval=1000 / FPS, blit=False)
anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))
plt.close(fig)

print(f"Saved rotating GIF to {GIF_PATH.resolve()}")

In [ ]:
"""
Plot a robot trajectory from a delta-position log file — Jupyter-friendly version.

Expected file format (whitespace-separated, no header):
    diffx  diffy  diffz  q1  q2  q3  q4  speed

Each row gives the CHANGE in position for that step (relative to the
previous point), starting from an assumed origin of (0, 0, 0). The actual
trajectory is the cumulative sum of the diffx/diffy/diffz columns.

Just edit FILEPATH below and run the cell.
"""



# ---- Configuration ----
FILEPATH = Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\Matlab\ABB_control\abb_trajectories\sine_5x5y_s3_zamp05.txt")  # <- change this to your file
COLOR_BY = "index"  # "index" or "speed"

# ---- Load data ----
data = np.loadtxt(FILEPATH)
if data.ndim == 1:
    data = data.reshape(1, -1)

diffx, diffy, diffz = data[:, 0], data[:, 1], data[:, 2]
speed = data[:, 7] if data.shape[1] > 7 else None

# Actual position = cumulative sum of the per-step deltas, starting at (0,0,0)
x = np.concatenate(([0.0], np.cumsum(diffx)))
y = np.concatenate(([0.0], np.cumsum(diffy)))
z = np.concatenate(([0.0], np.cumsum(diffz)))

# ---- Plot ----
fig = plt.figure(figsize=(14, 6))

# 3D view
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
if COLOR_BY == "speed" and speed is not None:
    c = np.concatenate(([speed[0]], speed))
    label = "speed"
else:
    c = np.arange(len(x))
    label = "step index"

sc = ax3d.scatter(x, y, z, c=c, cmap="viridis", s=8)
fig.colorbar(sc, ax=ax3d, shrink=0.6, label=label)
ax3d.plot(x, y, z, color="gray", linewidth=0.6, alpha=0.6)
ax3d.scatter(x[0], y[0], z[0], color="green", s=60, label="start")
ax3d.scatter(x[-1], y[-1], z[-1], color="red", s=60, label="end")
ax3d.set_xlabel("X")
ax3d.set_ylabel("Y")
ax3d.set_zlabel("Z")
ax3d.set_title("3D trajectory")
ax3d.legend()

# Top-down XY view
ax2d = fig.add_subplot(1, 2, 2)
sc2 = ax2d.scatter(x, y, c=np.arange(len(x)), cmap="viridis", s=8)
ax2d.plot(x, y, color="gray", linewidth=0.6, alpha=0.6)
ax2d.scatter(x[0], y[0], color="green", s=60, label="start", zorder=5)
ax2d.scatter(x[-1], y[-1], color="red", s=60, label="end", zorder=5)
ax2d.set_xlabel("X")
ax2d.set_ylabel("Y")
ax2d.set_title("Top-down view (XY)")
ax2d.set_aspect("equal", adjustable="box")
ax2d.legend()
fig.colorbar(sc2, ax=ax2d, shrink=0.8, label="step index")

fig.suptitle(FILEPATH.name)
fig.tight_layout()
plt.show()

### Plot iipp

In [67]:
# Download map dfs
map_df_paths = list(pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20").glob('*.csv'))
map_df_paths = [p for p in map_df_paths if '_Classes3_' in p.__str__()]
[print(p) for p in map_df_paths]
lab_dfs = pd.concat([pd.read_csv(p) for p in map_df_paths], axis=0).copy()
lab_dfs['img_relative_path'] = [pathlib.Path(p) for p in lab_dfs['img_relative_path']]
lab_dfs.loc[:, 'area'] = [p.parent.name.split('_')[-1] for p in lab_dfs['img_relative_path']]
lab_dfs['area_id'] = [int(re.sub(r'[^\d]+', '', a) ) for a in lab_dfs['area']]
lab_dfs['traj'] = ['_'.join(p.stem.split('_')[-6:-3]) for p in lab_dfs['img_relative_path']]
lab_dfs = lab_dfs[lab_dfs['traj'].str.contains('s8')].copy()
lab_dfs

C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20\test_mapping_Classes3_Sine_512scans.csv
C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20\train_mapping_Classes3_Sine_512scans.csv
C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data\512mscans_noNoise_sample20\valid_mapping_Classes3_Sine_512scans.csv


,img_relative_path,label,old_label,idx_start,idx_end,subset,area,area_id,traj
727,512mscans_noNoise_sample20\chicken_heart_p2_ar...,chicken_heart_muscle,chicken_heart,1,512,test,area6,6,sine_5x5y_s8
728,512mscans_noNoise_sample20\chicken_heart_p2_ar...,chicken_heart_muscle,chicken_heart,513,1024,test,area6,6,sine_5x5y_s8
729,512mscans_noNoise_sample20\chicken_heart_p2_ar...,chicken_heart_muscle,chicken_heart,1025,1536,test_supervised,area6,6,sine_5x5y_s8
730,512mscans_noNoise_sample20\chicken_heart_p2_ar...,chicken_heart_muscle,chicken_heart,1537,2048,test,area6,6,sine_5x5y_s8
731,512mscans_noNoise_sample20\chicken_heart_p2_ar...,chicken_heart_muscle,chicken_heart,2049,2560,test,area6,6,sine_5x5y_s8
...,...,...,...,...,...,...,...,...,...
4618,512mscans_noNoise_sample20\lamb_testicle_area3...,lamb_testicle,lamb_testicle,115713,116224,valid,area3,3,sine_5x5y_s8
4619,512mscans_noNoise_sample20\lamb_testicle_area3...,lamb_testicle,lamb_testicle,116225,116736,valid_supervised,area3,3,sine_5x5y_s8
4620,512mscans_noNoise_sample20\lamb_testicle_area3...,lamb_testicle,lamb_testicle,116737,117248,valid,area3,3,sine_5x5y_s8
4621,512mscans_noNoise_sample20\lamb_testicle_area3...,lamb_testicle,lamb_testicle,117249,117760,valid,area3,3,sine_5x5y_s8


In [69]:
# Filter to keep only data from some areas
combos_to_keep = {'chicken_heart_muscle': [1, 2],
                  'lamb_heart_fat': [1], 
                  'lamb_heart_muscle': [5],
                  # 'lamb_liver': [10],
                  'lamb_testicle': [1]
                  }
mask = lab_dfs.apply(lambda row: row['area_id'] in combos_to_keep.get(row['label'], []), axis=1)
lab_dfs = lab_dfs[mask].copy()

# Keep first two images for each area
lab_dfs_first = lab_dfs.sort_values(['label', 'area', 'idx_start']).groupby(['label', 'area']).head(2)

# Keep extra random 3 images for each area
lab_dfs_extra = lab_dfs.sort_values(['label', 'area', 'idx_start']).groupby(['label', 'area']).sample(3)

# Concat to get images to plot
lab_dfs_plt = pd.concat([lab_dfs_first, lab_dfs_extra], axis=0).sort_values(['label', 'area', 'idx_start'])
lab_dfs_plt

,img_relative_path,label,old_label,idx_start,idx_end,subset,area,area_id,traj
433,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,1,512,train,area1,1,sine_5x5y_s8
434,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,513,1024,train,area1,1,sine_5x5y_s8
438,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,2561,3072,train,area1,1,sine_5x5y_s8
454,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,10753,11264,train,area1,1,sine_5x5y_s8
509,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,38913,39424,train_supervised,area1,1,sine_5x5y_s8
1333,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,1,512,train,area2,2,sine_5x5y_s8
1334,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,513,1024,train,area2,2,sine_5x5y_s8
1365,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,16385,16896,train,area2,2,sine_5x5y_s8
1450,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,59905,60416,train,area2,2,sine_5x5y_s8
1467,512mscans_noNoise_sample20\chicken_heart_p1_ar...,chicken_heart_muscle,chicken_heart,68609,69120,train,area2,2,sine_5x5y_s8


In [77]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# Configuration
# ============================================================
img_root = pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_lab_data")

N_ROWS = 5
IMAGE_HEIGHT = 503
IMAGE_WIDTH = 512

# Size of the subtitle text
SUBTITLE_FONTSIZE = 16

# Controls how much vertical space is allocated to subtitles.
# Smaller = subtitles take up less space.
SUBTITLE_HEIGHT = 0.25

# Controls the width of the "..." separator columns.
# Smaller = less horizontal space around the ellipses.
SEPARATOR_WIDTH = 0.04

# Horizontal spacing between all GridSpec columns
COLUMN_SPACING = 0.10

# Vertical spacing between rows
ROW_SPACING = 0.15

# Subtitles for each row
row_titles = [
    "Chicken heart - Area 1",
    "Chicken heart - Area 2",
    "Lamb heart fat - Area 1",
    "Lamb heart muscle - Area 1",
    "Lamb testicle - Area 1",
]

# ============================================================
# Figure and layout
# ============================================================

fig = plt.figure(figsize=(15, 15))

# Each image row:
#
#   image | image | ... | image | ... | image | ... | image
#
# The first two images have normal spacing.
# Ellipses occur between images 2–3, 3–4, and 4–5.

gs = GridSpec(
    nrows=N_ROWS * 2,
    ncols=8,
    figure=fig,

    width_ratios=[
        1,                  # image 1
        1,                  # image 2
        SEPARATOR_WIDTH,    # ...
        1,                  # image 3
        SEPARATOR_WIDTH,    # ...
        1,                  # image 4
        SEPARATOR_WIDTH,    # ...
        1                   # image 5
    ],

    # Alternating subtitle/image rows
    height_ratios=[
        SUBTITLE_HEIGHT, 1,
        SUBTITLE_HEIGHT, 1,
        SUBTITLE_HEIGHT, 1,
        SUBTITLE_HEIGHT, 1,
        SUBTITLE_HEIGHT, 1,
    ],

    hspace=ROW_SPACING,
    wspace=COLUMN_SPACING
)

image_columns = [0, 1, 3, 5, 7]
separator_columns = [2, 4, 6]

# ============================================================
# Plot rows
# ============================================================

for row in range(N_ROWS):

    subtitle_row = row * 2
    image_row = row * 2 + 1

    # --------------------------------------------------------
    # Subtitle
    # --------------------------------------------------------

    ax_title = fig.add_subplot(
        gs[subtitle_row, :]
    )

    ax_title.text(
        0.5,
        0.5,
        row_titles[row],
        ha="center",
        va="center",
        fontsize=SUBTITLE_FONTSIZE
    )

    ax_title.set_axis_off()

    # --------------------------------------------------------
    # Images
    # --------------------------------------------------------

    for image_number, col in enumerate(image_columns):

        # ----------------------------------------------------
        # Get image path from dataframe
        # ----------------------------------------------------

        # Assumes each row of lab_dfs corresponds to one
        # image in the grid.
        #
        # Adjust this indexing if your dataframe is structured
        # differently.

        df_index = row * 5 + image_number

        img_path = img_root.joinpath(lab_dfs_plt.iloc[df_index]["img_relative_path"])
        image = cv2.imread(img_path)
        # image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_CUBIC)

        # ----------------------------------------------------
        # Temporary placeholder image
        # ----------------------------------------------------

        # Currently generate a random image instead of reading
        # img_path.

        # np.random.seed(df_index)
        # image = np.random.rand(
        #     IMAGE_HEIGHT,
        #     IMAGE_WIDTH
        # )

        # ----------------------------------------------------
        # Later, replace the above with something like:
        #
        # image = plt.imread(img_path)
        #
        # or, for example:
        #
        # from PIL import Image
        # image = np.array(Image.open(img_path))
        # ----------------------------------------------------

        ax = fig.add_subplot(
            gs[image_row, col]
        )

        ax.imshow(
            image,
            cmap="gray",
            interpolation="nearest"
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

    # --------------------------------------------------------
    # Ellipses
    # --------------------------------------------------------

    for col in separator_columns:

        ax = fig.add_subplot(
            gs[image_row, col]
        )

        ax.text(
            0.5,
            0.5,
            "...",
            ha="center",
            va="center",
            fontsize=14
        )

        ax.set_axis_off()

# ============================================================
# Final adjustments
# ============================================================

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.98,
    bottom=0.02
)

plt.savefig("iipp_example.png")
plt.show()



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# Configuration
# ============================================================

N_ROWS = 5
IMAGE_HEIGHT = 256 # 503
IMAGE_WIDTH = 256 # 512

# Generate one random image as a placeholder
np.random.seed(42)
image = np.random.rand(IMAGE_HEIGHT, IMAGE_WIDTH)

# Subtitles for each row
row_titles = [
    "Row 1 subtitle",
    "Row 2 subtitle",
    "Row 3 subtitle",
    "Row 4 subtitle",
    "Row 5 subtitle",
]

# ============================================================
# Figure and layout
# ============================================================

fig = plt.figure(figsize=(15, 15))

# Each row consists of:
#
#   image | image | ... | image | ... | image | ... | image
#
# The first two images have normal spacing.
# Ellipses occur between images 2–3, 3–4, and 4–5.
#
# A separate row above each image row is reserved for the
# corresponding subtitle.

# Each image row uses 8 columns:
#   0 = image 1
#   1 = image 2
#   2 = ...
#   3 = image 3
#   4 = ...
#   5 = image 4
#   6 = ...
#   7 = image 5

# Add one extra column structure for the subtitle.
gs = GridSpec(
    nrows=N_ROWS * 2,
    ncols=8,
    figure=fig,

    # Smaller values make the "..." columns narrower.
    width_ratios=[
        1,      # image 1
        1,      # image 2
        0.04,   # ...
        1,      # image 3
        0.04,   # ...
        1,      # image 4
        0.04,   # ...
        1       # image 5
    ],

    # Alternating subtitle/image rows
    height_ratios=[
        0.12, 1,
        0.12, 1,
        0.12, 1,
        0.12, 1,
        0.12, 1,
    ],

    hspace=0.15,
    wspace=0.10
)

image_columns = [0, 1, 3, 5, 7]
separator_columns = [2, 4, 6]

for row in range(N_ROWS):

    subtitle_row = row * 2
    image_row = row * 2 + 1

    # --------------------------------------------------------
    # Subtitle
    # --------------------------------------------------------

    # Use the entire row for the subtitle so that it is
    # centered with respect to the complete five-image row.

    ax_title = fig.add_subplot(
        gs[subtitle_row, :]
    )

    ax_title.text(
        0.5,
        0.5,
        row_titles[row],
        ha="center",
        va="center",
        fontsize=16
    )

    ax_title.set_axis_off()

    # --------------------------------------------------------
    # Images
    # --------------------------------------------------------

    for col in image_columns:

        ax = fig.add_subplot(
            gs[image_row, col]
        )

        ax.imshow(
            image,
            cmap="gray",
            interpolation="nearest"
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

    # --------------------------------------------------------
    # Ellipses
    # --------------------------------------------------------

    for col in separator_columns:

        ax = fig.add_subplot(
            gs[image_row, col]
        )

        ax.text(
            0.5,
            0.5,
            "...",
            ha="center",
            va="center",
            fontsize=20
        )

        ax.set_axis_off()

# ============================================================
# Final adjustments
# ============================================================

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.98,
    bottom=0.02
)

plt.show()
